# Strained CALPHAD model for wurtzite Al$_{1-x}$Sc$_x$N

Built on a Python reimplementation of the published assessment:

K. Hirata, K. Shobu, H. Yamada, M. Uehara, S. A. Anggraini, M. Akiyama,
*"Thermodynamic assessment of the Al–Sc–N ternary system and phase-separated region of the
strained wurtzite phase,"* **J. Eur. Ceram. Soc. 40 (2020) 5410–5422**,
doi:[10.1016/j.jeurceramsoc.2020.06.047](https://doi.org/10.1016/j.jeurceramsoc.2020.06.047)

Sections 2–15 reproduce and validate the published database and recover the data it only plots
(same content as `Hirata2020_AlScN_CALPHAD.ipynb`, with the rocksalt endmembers added and the
binodal computed by a true common tangent). Sections 16–22 add the strain model.

Energies are in meV per formula unit (f.u. = Al$_{1-x}$Sc$_x$N) in sections 16–22 and in all plots.

Requires `numpy` and `scipy`; `matplotlib` is optional and only used for plots.

---

## Strain model (sections 16–22)

| | |
|---|---|
| **Inputs** | $a(x)$, $c(x)$ from the continuity-constrained fit; $E(x)$, $\nu(x)$, $V_m(x)$ from the recovered Fig. 10–11 data; no new DFT |
| **Anisotropic misfit** | $a$ and $c$ misfits enter separately; the interface orientation is minimised, as in Doak & Wolverton (2012) and Müller *et al.* (2026). Moduli stay isotropic (VRH) |
| **Cases** | `incoherent` (Hirata bulk); `coherent` (free-standing, domains coherent with each other); `clamped` (film on a substrate, layered or columnar domains). One decomposition (binodal) and one stability (spinodal) function per case, taking $G(x,T)$, $T$, $\bar x$ and the substrate lattice parameter |
| **Outputs** | critical temperatures, gap boundaries, decomposition driving force at growth temperatures, the strain ↔ $L^{(0)}$ identifiability, and a $G_\mathrm{mix}(x,T)$ function for the defect model |
| **Optional, off by default** | analytic short-range-order correction; guessed excess vibrational entropy |
| **Not included** | hexagonal elastic constants (isotropic surrogate; placeholder for Zhang *et al.* 2013), anharmonic epitaxial energies, three-domain states, interface energy, an explicit h-BN phase or order parameter |

## Headline findings

1. **The wurtzite critical temperature is 6972 K at $x_c \approx 0.50$**, once the sign of $L^{(2)}$ lost from
   the PDF text is fixed from the paper's own Fig. 14 (section 6a); the regular-solution shortcut
   $L^{(0)}/2R$ = 5892 K understates it.
2. **Table 2 and Table 3 disagree on the AlN wurtzite formation enthalpy by 8.50 kJ/mol-atom (5.4 %)**;
   the discrepancy sits entirely in the leading constant.


## 2. Conventions: units and the per-formula-unit vs per-atom trap

CALPHAD mixes *per mole of formula unit* with *per mole of atoms*, and this is the single largest
source of factor-of-two errors when reading this paper. Conventions used throughout:

- Table 2 is **"J/mol of model"**. For wurtzite the model is `(Al,Sc)(N)` = **1 formula unit = 2 atoms**.
- Table 3 labels its column **"kJ/mol of atom"** — correct.
- Table 4 heads its formation-enthalpy block **"kJ/mol"** — **wrong by a factor of two**; the value
  −185.5 only matches the Table 2 Gibbs energy on a *per-atom* reading.

Every quantity below carries its basis in the variable name.

In [ ]:
import dataclasses
import functools
import math
import operator

import numpy as np
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq, minimize
from scipy.special import xlogy

GAS_CONSTANT = 8.314462618            # J / (mol K)
ROOM_TEMPERATURE = 298.15             # K
JOULE_PER_ELECTRONVOLT = 96485.33212  # J/mol per eV/atom

ATOMS_PER_FORMULA_UNIT_WURTZITE = 2   # (Al,Sc)(N)
ATOMS_PER_FORMULA_UNIT_ROCKSALT = 2   # (Al,Sc)(N,Va) on the N-full end


def joules_per_formula_unit_to_millielectronvolt_per_atom(value_joule_per_formula_unit,
                                                          atoms_per_formula_unit=2):
    """J/mol-formula-unit -> meV/atom."""
    return (value_joule_per_formula_unit / atoms_per_formula_unit
            / JOULE_PER_ELECTRONVOLT * 1000.0)

## 3. Cross-check harness

Every quantity that is **both** stated in the paper **and** derivable from other published values is
recomputed from the raw parameters and asserted against the stated value. Three checks are
**expected to fail** — they are the internal inconsistencies of §9, and they are marked
`expect_failure=True` so a clean run still reports `ALL AS EXPECTED`.

In [ ]:
CHECK_RESULTS = []


def check_value(computed, expected, tolerance_relative, label,
                expect_failure=False, note=""):
    """Assert computed ~= expected to a relative tolerance; record PASS/FAIL."""
    if expected == 0:
        relative_error = abs(computed)
    else:
        relative_error = abs(computed - expected) / abs(expected)
    passed = relative_error <= tolerance_relative
    as_expected = (passed != expect_failure)
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": expected,
                          "relative_error": relative_error, "passed": passed,
                          "expect_failure": expect_failure, "as_expected": as_expected,
                          "note": note})
    tag = "PASS" if passed else "FAIL"
    if expect_failure:
        tag += " (expected FAIL)" if not passed else " (UNEXPECTED PASS)"
    print(f"[{tag:22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  expected={expected:.6g}  rel.err={relative_error:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def check_zero(computed, absolute_tolerance, label, note=""):
    passed = abs(computed) <= absolute_tolerance
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": 0.0,
                          "relative_error": abs(computed), "passed": passed,
                          "expect_failure": False, "as_expected": passed, "note": note})
    print(f"[{'PASS' if passed else 'FAIL':22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  |tol|={absolute_tolerance:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def summarise_checks():
    total = len(CHECK_RESULTS)
    as_expected = sum(1 for r in CHECK_RESULTS if r["as_expected"])
    print(f"\n{'='*78}\n{as_expected}/{total} checks behaved as expected.")
    unexpected = [r for r in CHECK_RESULTS if not r["as_expected"]]
    if unexpected:
        print("UNEXPECTED:")
        for r in unexpected:
            print(f"  - {r['label']}  (rel.err {r['relative_error']:.3g})")
    else:
        print("ALL AS EXPECTED (including the two deliberate failures of section 9).")

## 4. Table 1 — SGTE lattice stabilities, and proving the sign reconstruction

Standard SGTE polynomial, J per mole of atoms:

$$G(T) = a + bT + cT\ln T + dT^{2} + eT^{-1} + fT^{3}$$

from which

$$S = -\frac{dG}{dT}, \qquad H = G - T\frac{dG}{dT} = a - cT - dT^{2} + \frac{2e}{T} - 2fT^{3}$$

**Why this section matters.** The PDF text layer drops every minus glyph, so all signs had to be
reconstructed. The reconstruction is *provable*: for the SGTE reference states,
`H(298.15) − H_SER` must be **exactly zero** and `S(298.15)` must match the tabulated standard
entropies — neither of which was used as an input.

In [ ]:
def sgte_gibbs_energy(coefficients, temperature):
    """G(T) = a + b T + c T lnT + d T^2 + e/T + f T^3   [J per mol of atoms]."""
    a, b, c, d, e, f = coefficients
    return (a + b * temperature + c * temperature * math.log(temperature)
            + d * temperature ** 2 + e / temperature + f * temperature ** 3)


def sgte_entropy(coefficients, temperature):
    a, b, c, d, e, f = coefficients
    return -(b + c * (math.log(temperature) + 1.0) + 2.0 * d * temperature
             - e / temperature ** 2 + 3.0 * f * temperature ** 2)


def sgte_enthalpy(coefficients, temperature):
    a, b, c, d, e, f = coefficients
    return (a - c * temperature - d * temperature ** 2
            + 2.0 * e / temperature - 2.0 * f * temperature ** 3)


# ---- SGTE reference states (Dinsdale 1991), 298.15 K < T < 700 / 800 K branches ----
# (a, b, c, d, e, f)
GIBBS_ALUMINIUM_FCC_LOW = (-7976.15, 137.093038, -24.3671976,
                           -1.884662e-3, 74092.0, -0.877664e-6)
GIBBS_SCANDIUM_HCP_LOW = (-8689.547, 153.48097, -28.1882,
                          +3.21892e-3, 72177.0, -1.64531e-6)
GIBBS_HALF_NITROGEN_GAS_LOW = (-3750.675, -9.45425, -12.7819,
                               -1.76686e-3, -32374.0, 2.681e-9)

print("SGTE reference states at 298.15 K "
      "(H - H_SER must vanish; S must match the standard entropy)\n")
for name, coefficients, standard_entropy in [
        ("Al fcc  ", GIBBS_ALUMINIUM_FCC_LOW, 28.30),
        ("Sc hcp  ", GIBBS_SCANDIUM_HCP_LOW, 34.64),
        ("0.5 N2  ", GIBBS_HALF_NITROGEN_GAS_LOW, 191.61 / 2)]:
    enthalpy = sgte_enthalpy(coefficients, ROOM_TEMPERATURE)
    entropy = sgte_entropy(coefficients, ROOM_TEMPERATURE)
    print(f"  {name}  H-H_SER = {enthalpy:12.4f} J/mol      "
          f"S = {entropy:9.4f}  (SGTE {standard_entropy:.2f})")

## 5. Table 2 — the assessed parameters

The values that matter for everything downstream. Note the wurtzite ternary gets **three**
Redlich–Kister terms while the rocksalt (FCC) ternary gets **one** — an asymmetry that biases the
wurtzite/rocksalt crossover, since the crossover is a *difference* of two Gibbs curves.

Note also that **every ternary L is a bare constant with no temperature coefficient**, so the excess
entropy of mixing (i.e. the mixing-induced excess vibrational entropy) is identically zero.

In [ ]:
# --- Table 2, Al-Sc-N ternary. J per mol of model; model = (Al,Sc)(N) = 1 f.u. = 2 atoms
# NOTE on the sign of L^2: the PDF text layer drops every minus glyph, so all Table 2
# signs had to be reconstructed. L^2 is the one sign that Table 2 alone cannot settle.
# It is fixed here by digitizing the paper's OWN Fig. 14 (bulk miscibility gap) and
# comparing against both candidates -- see section 6a. L^2 < 0 reproduces Fig. 14 to a
# mean error of 172 K over 627 digitized points; L^2 > 0 gives 1207 K and the wrong shape.
REDLICH_KISTER_WURTZITE = (97970.4838186,     # L^0
                           434.322298892,     # L^1
                           -17960.9058372)    # L^2  <-- NEGATIVE (see section 6a)
REDLICH_KISTER_ROCKSALT = (90970.6587083,)    # L^0 only -- the paper fits no more

# --- Table 2, wurtzite endmembers. J per mol of formula unit.
# Signs reconstructed; proven below against Table 3's entropy.
GIBBS_ALUMINIUM_NITRIDE_WURTZITE = (-315445.36, 291.78238, -45.177271,
                                    -0.00285744685, 683276.5, 1.69215267e-17)
GIBBS_SCANDIUM_NITRIDE_WURTZITE = (-347780.61, 285.610224, -48.138732,
                                   -0.0011557218, 295515.2, 0.0)

# --- Table 2, rocksalt (FCC) endmembers, absolute G, J per mol of formula unit, 298.15-5000 K.
# Signs of the T^2 and 1/T coefficients fixed by C_p > 0 and increasing; ScN reproduces
# Table 4's -185.5 kJ/mol-atom to 0.02 %.
GIBBS_ALUMINIUM_NITRIDE_ROCKSALT = (-260285.275, 303.149378, -45.580547,
                                    -0.00186786345, 982780.85, 0.0)
GIBBS_SCANDIUM_NITRIDE_ROCKSALT = (-388602.46, 289.579582, -46.593617,
                                   -0.00248812465, 530396.55, 0.0)

# --- Sc-N rocksalt (FCC) vacancy interaction -- the ONLY vacancy modelling in the paper
REDLICH_KISTER_SCANDIUM_NITRIDE_FCC_VACANCY = (20606.0368145, 22501.84214)

# --- Tables 3 and 4, as printed
ALUMINIUM_NITRIDE_ENTHALPY_TABLE3_KJ_PER_MOL_ATOM = -157.07
ALUMINIUM_NITRIDE_ENTROPY_TABLE3 = 10.07          # J/(mol-atom K)
SCANDIUM_NITRIDE_ENTHALPY_TABLE4 = -185.5         # kJ/mol -- per ATOM despite the label
SCANDIUM_NITRIDE_LATTICE_CONSTANT_NM = 0.452
ALUMINIUM_SCANDIUM_NITRIDE_LATTICE_CONSTANT_NM = 0.442

print("wurtzite  L0, L1, L2 =", REDLICH_KISTER_WURTZITE, "J/mol-f.u.")
print("rocksalt  L0         =", REDLICH_KISTER_ROCKSALT, "J/mol-f.u.")

## 6. Redlich–Kister mixing, and the real critical point

For $\mathrm{Al}_{1-x}\mathrm{Sc}_x\mathrm{N}$ with $y_{\mathrm{Al}} = 1-x$, $y_{\mathrm{Sc}} = x$:

$$G^{\mathrm{ex}}(x) = x(1-x)\sum_{n} L^{(n)}\,(1-2x)^{n}$$
$$G^{\mathrm{mix}}(x,T) = G^{\mathrm{ex}}(x) + RT\big[x\ln x + (1-x)\ln(1-x)\big]$$

The critical point satisfies $\partial^2 G/\partial x^2 = \partial^3 G/\partial x^3 = 0$.

**The regular-solution shortcut does not apply to the wurtzite phase.** $T_c = L^{(0)}/2R$ is
exact only for a strictly *regular* solution, $L^{(1)}=L^{(2)}=0$. That holds for the **rocksalt**
phase here — only $L^{(0)}$ was fitted, so its 5471 K is exact — but not for wurtzite, where
$|L^{(2)}|$ is a fifth of $L^{(0)}$.

Because $L^{(2)}$ is **negative** (section 6a), its term *sharpens* the top of the dome rather than
flattening it. The critical point stays essentially at $x=0.5$ and $T_c$ rises **above** the
regular-solution estimate:

| | $T_c$ | $x_c$ |
|---|---|---|
| regular-solution shortcut $L^{(0)}/2R$ | 5892 K | 0.5 (assumed) |
| **full three-term RK** | **6972 K** | **0.4985** |

Note the parity structure: the term is $x(1-x)(1-2x)^n$, so **even $n$ is symmetric about $x=0.5$
and odd $n$ is antisymmetric**. $L^{(1)}$ is therefore a pure skew term and $L^{(2)}$ a symmetric
shape term. Here $L^{(1)}/L^{(0)} = 0.44\%$ while $|L^{(2)}|/L^{(0)} = 18.3\%$, so the gap is very
nearly symmetric and essentially all of the non-parabolic character comes from $L^{(2)}$.

Both $(1-2x)$ and $(1-2x)^2$ vanish at $x=0.5$, so $\Delta H_{mix}(0.5) = L^{(0)}/4$ **exactly**,
independent of $L^{(1)}$ and $L^{(2)}$.

In [ ]:
def excess_gibbs_energy(scandium_fraction, redlich_kister_coefficients):
    """Redlich-Kister excess Gibbs energy, J per mol of formula unit."""
    x = scandium_fraction
    asymmetry = 1.0 - 2.0 * x
    series = sum(coefficient * asymmetry ** order
                 for order, coefficient in enumerate(redlich_kister_coefficients))
    return x * (1.0 - x) * series


def ideal_mixing_entropy(scandium_fraction):
    """Configurational entropy on the CATION sublattice only, J/(mol-f.u. K).

    Only the cation sublattice mixes, so per ATOM this is half the usual R ln2 --
    0.347 kB/atom, not 0.693. That halving is why the excess VIBRATIONAL entropy of
    mixing (0.1-0.2 kB/atom in substitutional alloys) is such a large fraction of the
    total here, and why omitting it matters.
    """
    x = scandium_fraction
    if x <= 0.0 or x >= 1.0:
        return 0.0
    return -GAS_CONSTANT * (x * math.log(x) + (1.0 - x) * math.log(1.0 - x))


def gibbs_energy_of_mixing(scandium_fraction, temperature, redlich_kister_coefficients):
    return (excess_gibbs_energy(scandium_fraction, redlich_kister_coefficients)
            - temperature * ideal_mixing_entropy(scandium_fraction))


def second_derivative_of_mixing(scandium_fraction, temperature,
                                redlich_kister_coefficients, step=1e-5):
    x, h = scandium_fraction, step
    return (gibbs_energy_of_mixing(x + h, temperature, redlich_kister_coefficients)
            - 2.0 * gibbs_energy_of_mixing(x, temperature, redlich_kister_coefficients)
            + gibbs_energy_of_mixing(x - h, temperature, redlich_kister_coefficients)) / h ** 2


def spinodal_interval(temperature, redlich_kister_coefficients, sample_count=20001):
    """Composition range over which d2G/dx2 < 0, or None if the alloy is stable."""
    unstable = [1e-4 + (1.0 - 2e-4) * i / (sample_count - 1)
                for i in range(sample_count)
                if second_derivative_of_mixing(1e-4 + (1.0 - 2e-4) * i / (sample_count - 1),
                                               temperature, redlich_kister_coefficients) < 0.0]
    return (min(unstable), max(unstable)) if unstable else None


def critical_temperature(redlich_kister_coefficients,
                         lower_bound=1000.0, upper_bound=9000.0, iterations=60):
    """Highest T with any spinodal region, plus the composition where it closes."""
    low, high = lower_bound, upper_bound
    for _ in range(iterations):
        middle = 0.5 * (low + high)
        if spinodal_interval(middle, redlich_kister_coefficients, 4001):
            low = middle
        else:
            high = middle
    interval = spinodal_interval(low, redlich_kister_coefficients, 200001)
    critical_composition = 0.5 * (interval[0] + interval[1]) if interval else float("nan")
    return low, critical_composition


wurtzite_critical_temperature, wurtzite_critical_composition = \
    critical_temperature(REDLICH_KISTER_WURTZITE)
regular_solution_estimate = REDLICH_KISTER_WURTZITE[0] / (2.0 * GAS_CONSTANT)
rocksalt_critical_temperature = REDLICH_KISTER_ROCKSALT[0] / (2.0 * GAS_CONSTANT)

print(f"wurtzite, FULL three-term RK : T_c = {wurtzite_critical_temperature:7.1f} K "
      f"at x_c = {wurtzite_critical_composition:.4f}")
print(f"wurtzite, regular-solution   : T_c = {regular_solution_estimate:7.1f} K at x   = 0.5"
      f"   <-- overstates by "
      f"{(regular_solution_estimate/wurtzite_critical_temperature-1)*100:.0f}%")
print(f"rocksalt, L0 only (exact)    : T_c = {rocksalt_critical_temperature:7.1f} K at x   = 0.5")
print("\nSpinodal interval versus temperature:")
for temperature in (3000, 4000, 4500, 4900, 4940, 4944, 5000, 5892):
    interval = spinodal_interval(temperature, REDLICH_KISTER_WURTZITE)
    print(f"  T = {temperature:5d} K  ->  "
          + (f"{interval[0]:.4f} .. {interval[1]:.4f}" if interval else "single phase"))

## 6a. Determining the sign of $L^{(2)}$ from the paper's own Fig. 14

The PDF text layer drops every minus glyph, so all Table 2 signs are reconstructions. Most are
provable — section 4 shows $H-H_{SER}=0$ and $S(298.15)$ recovering SGTE reference values that were
never inputs. **$L^{(2)}$ is the one that Table 2 alone cannot settle**, because it does not affect
$\Delta H_{mix}(0.5) = L^{(0)}/4$ and contributes nothing at the endpoints.

Fig. 14 (bulk miscibility gap) is an independent constraint. Digitizing its bulk curve and comparing
against both sign hypotheses is decisive:

| $L^{(2)}$ | $T_c$ | $x_c$ | mean abs. error vs 627 digitized points |
|---|---|---|---|
| $+17960.9$ | 4944 K | 0.317 | **1207 K** — wrong shape (double dome, apex off-centre) |
| $\mathbf{-17960.9}$ | **6972 K** | **0.4985** | **172 K** ✓ |

The check below reproduces that comparison. For a near-symmetric system the binodal is simply where
$dG_{mix}/dx = 0$ (the two minima of the mixing free energy), which inverts analytically for $T$.

In [ ]:
def binodal_temperature(scandium_fraction, redlich_kister_coefficients, step=1e-7):
    """Temperature at which a composition lies on the binodal, for a near-symmetric system.

    Binodal = the two minima of G_mix, i.e. dG/dx = 0, which solves directly for T.
    Exact when the excess is symmetric; L^1/L^0 = 0.4% here, so the error is negligible.
    """
    x = scandium_fraction
    if abs(x - 0.5) < 1e-6 or x <= 0.0 or x >= 1.0:
        return float("nan")
    forward = excess_gibbs_energy(x + step, redlich_kister_coefficients)
    backward = excess_gibbs_energy(x - step, redlich_kister_coefficients)
    excess_slope = (forward - backward) / (2.0 * step)
    return -excess_slope / (GAS_CONSTANT * math.log(x / (1.0 - x)))


# Bulk binodal digitized from Fig. 14 (this work, 627 points; +-60 K reading uncertainty,
# set by the 3 px curve thickness against a 7000 K full scale over 666 px).
# Columns where the "Bulk"/"h=100nm" text labels overlap the curve are excluded.
FIG14_BULK_BINODAL = [   # (cation x, temperature K)
    (0.101, 3968), (0.150, 4678), (0.200, 5272), (0.249, 5771), (0.300, 6197),
    (0.350, 6507), (0.399, 6743), (0.450, 6880), (0.550, 6890), (0.580, 6812),
    (0.850, 4709), (0.899, 4016),
]

print("Sign of L^2, tested against the digitized Fig. 14 bulk binodal\n")
print(f"{'x':>7} {'T_fig':>7} {'L2<0':>7} {'err':>6} | {'L2>0':>7} {'err':>6}")
error_negative = error_positive = 0.0
for composition, figure_temperature in FIG14_BULK_BINODAL:
    negative = binodal_temperature(composition, (97970.4838186, 434.322298892, -17960.9058372))
    positive = binodal_temperature(composition, (97970.4838186, 434.322298892, +17960.9058372))
    error_negative += abs(figure_temperature - negative)
    error_positive += abs(figure_temperature - positive)
    print(f"{composition:7.3f} {figure_temperature:7.0f} {negative:7.0f} "
          f"{figure_temperature - negative:6.0f} | {positive:7.0f} {figure_temperature - positive:6.0f}")
count = len(FIG14_BULK_BINODAL)
print(f"\nmean |error|:   L2 < 0 -> {error_negative / count:5.0f} K"
      f"      L2 > 0 -> {error_positive / count:5.0f} K")
print("=> L^2 is NEGATIVE. Adopted in section 5.")

## 6b. Binodal and spinodal — and which one each paper actually reports

These are different curves and the three sources do not compute the same thing:

| | binodal (common tangent) | spinodal ($d^2G/dx^2=0$) |
|---|---|---|
| **Hirata** | **yes** — Fig. 14, one line per thickness | no — the word never appears |
| **Talley (PRM 2, 063802)** | yes — Fig. 4 | yes (chemical) |
| **this notebook** | **yes** (below) | **yes** |

They share a critical point — the two curves are tangent at the apex — but separate below it, the
binodal being the wider. Neither paper computes the **coherent** spinodal (Cahn), which adds an
elastic term $2\eta^2 Y$ to $d^2G/dx^2$ and can be the same order as the entire chemical driving
force in a system this size-mismatched. It is implemented in sections 17–19.

The binodal below is the **lower convex hull** of $G_\mathrm{mix}$ — a true common tangent, valid for any shape of $G$.

In [ ]:
def spinodal_composition_range(temperature, redlich_kister_coefficients,
                               samples=4001, coherency_term=0.0):
    """Composition interval where d2G/dx2 + coherency_term < 0, or None if stable.

    coherency_term = 2*eta^2*Y*V_m in J/mol adds Cahn's coherent-spinodal contribution;
    leave 0 for the chemical spinodal, which is what both papers report.
    """
    unstable = []
    for index in range(samples):
        x = 1e-4 + (1.0 - 2e-4) * index / (samples - 1)
        if second_derivative_of_mixing(x, temperature, redlich_kister_coefficients) \
                + coherency_term < 0.0:
            unstable.append(x)
    return (min(unstable), max(unstable)) if unstable else None

COMPOSITION_MESH = np.linspace(0.0, 1.0, 4001)


def lower_hull_indices(compositions, energies):
    """Indices of the lower convex hull of points sorted by composition (monotone chain)."""
    hull = []
    for index in range(len(compositions)):
        while len(hull) >= 2:
            first, second = hull[-2], hull[-1]
            cross = ((compositions[second] - compositions[first]) * (energies[index] - energies[first])
                     - (energies[second] - energies[first]) * (compositions[index] - compositions[first]))
            if cross > 0.0:
                break
            hull.pop()
        hull.append(index)
    return hull


def common_tangent_gaps(energies, compositions=COMPOSITION_MESH):
    """Two-phase intervals (x_left, x_right) of a free-energy curve sampled on the mesh."""
    hull = lower_hull_indices(compositions, energies)
    return [(compositions[left], compositions[right])
            for left, right in zip(hull[:-1], hull[1:]) if right - left > 2]


def binodal_composition_range(temperature, redlich_kister_coefficients):
    """Widest common-tangent gap of G_mix (lower convex hull), or None if single phase."""
    energies = np.array([gibbs_energy_of_mixing(x, temperature, redlich_kister_coefficients)
                         for x in COMPOSITION_MESH])
    gaps = common_tangent_gaps(energies)
    return max(gaps, key=lambda gap: gap[1] - gap[0]) if gaps else None


print("Wurtzite miscibility gap from the corrected parameters\n")
print(f"{'T (K)':>7} {'binodal':>18} {'spinodal':>18}")
for temperature in (2000, 3000, 4000, 5000, 6000, 6900, 7000):
    b = binodal_composition_range(temperature, REDLICH_KISTER_WURTZITE)
    s = spinodal_composition_range(temperature, REDLICH_KISTER_WURTZITE)
    fmt = lambda r: f"{r[0]:.3f} .. {r[1]:.3f}" if r else "single phase"
    print(f"{temperature:7d} {fmt(b):>18} {fmt(s):>18}")

## 6c. The wurtzite/rocksalt crossover the paper never states

Hirata reports enthalpy curves (Fig. 8) and calls them "relatively consistent" with experiment
without ever quoting a crossover composition. It follows directly from Table 2: the crossover is
where the wurtzite and rocksalt Gibbs energies cross, endmembers plus mixing.

Context for the number this produces: **Zhang** (JAP 114, 243516) gets 0.56 without van der Waals;
**Talley** (PRM 2, 063802) gets 0.64 with PBE-D2 and explicitly corrects Zhang. Hirata used plain
PBE with no vdW correction.

**Important:** this equilibrium crossover is *not* the experimental "x ≈ 0.4 wurtzite limit."
That limit is kinetic — set by decomposition during growth, moves with substrate and temperature,
and Talley measures the rocksalt onset separately at x ≈ 0.65. Comparing the two is a category
error, and it is the one our own project's pre-registered metric has to avoid.

In [ ]:
def phase_gibbs_energy(scandium_fraction, temperature, aluminium_endmember,
                       scandium_endmember, redlich_kister_coefficients):
    """Total molar Gibbs energy of a two-sublattice (Al,Sc)(N) phase, J/mol-f.u."""
    x = scandium_fraction
    endmembers = ((1.0 - x) * sgte_gibbs_energy(aluminium_endmember, temperature)
                  + x * sgte_gibbs_energy(scandium_endmember, temperature))
    return (endmembers - temperature * ideal_mixing_entropy(x)
            + excess_gibbs_energy(x, redlich_kister_coefficients))


def wurtzite_rocksalt_crossover(temperature, samples=20001):
    """Composition where G(wurtzite) = G(rocksalt). Returns None if they do not cross."""
    previous_x = previous_difference = None
    for index in range(samples):
        x = 1e-6 + (1.0 - 2e-6) * index / (samples - 1)
        difference = (phase_gibbs_energy(x, temperature, GIBBS_ALUMINIUM_NITRIDE_WURTZITE,
                                         GIBBS_SCANDIUM_NITRIDE_WURTZITE,
                                         REDLICH_KISTER_WURTZITE)
                      - phase_gibbs_energy(x, temperature, GIBBS_ALUMINIUM_NITRIDE_ROCKSALT,
                                           GIBBS_SCANDIUM_NITRIDE_ROCKSALT,
                                           REDLICH_KISTER_ROCKSALT))
        if previous_difference is not None and previous_difference * difference < 0:
            span = previous_difference - difference
            return previous_x + (x - previous_x) * previous_difference / span
        previous_x, previous_difference = x, difference
    return None


for temperature in (298.15, 673.15, 1273.15):
    crossover = wurtzite_rocksalt_crossover(temperature)
    print(f"T = {temperature:7.2f} K   WZ/RS crossover at x = "
          + (f"{crossover:.3f}" if crossover else "no crossing"))
check_value(wurtzite_rocksalt_crossover(298.15), 0.60, 0.01,
            "WZ/RS crossover at 298 K from Table 2 (never stated in the paper)",
            note="0.603 with L^2 < 0; 0.599 with the uncorrected L^2 > 0")


## 7. Equations (20)–(22) — epitaxial strain energy

$$G_{\mathrm{strain}} = 2V\mu\frac{1+\nu}{1-\nu}\,\varepsilon^{2} \qquad (h < h_c)$$
$$G_{\mathrm{strain}} = 2V\mu\frac{1+\nu}{1-\nu}\,\varepsilon^{2}\cdot\frac{h_c}{h}\left(1+\ln\frac{h}{h_c}\right) \qquad (h > h_c)$$

None of $\mu(x)$, $\nu(x)$, $V(x)$, $\varepsilon(x)$ is tabulated (§8); the prefactor is evaluated in
section 17 from data recovered from the figures. The **relaxation factor** needs no paper-specific
inputs, and it is what carries the paper's central claim.

With $h_c \approx 2$ nm (Zhang *et al.*, JAP 114, 243516, for $\mathrm{Sc}_{0.375}\mathrm{Al}_{0.625}\mathrm{N}$
on AlN), all three thicknesses the paper models are deep in the dislocation-relaxed regime — and real
films are 1–2 µm.

### The analytic decomposition that matters for the strain/Ω degeneracy

With a linear Vegard misfit referenced to the substrate, $\varepsilon(x) = cx$, and $x^2 = x - x(1-x)$:

$$G_{\mathrm{strain}} = K(h)c^{2}x^{2} = \underbrace{K(h)c^{2}x}_{\text{endmember tilt}} - \underbrace{K(h)c^{2}\,x(1-x)}_{\Delta L^{(0)} = -K(h)c^{2}}$$

So epitaxial strain is *formally identical* to a thickness-dependent interaction parameter. Ω-calibration
and a strain term are therefore **degenerate against a single measured crossover composition**.

In [ ]:
def strain_relaxation_factor(film_thickness_nm, critical_thickness_nm):
    """Fraction of the fully-coherent strain energy retained at a given thickness."""
    if film_thickness_nm <= critical_thickness_nm:
        return 1.0
    ratio = critical_thickness_nm / film_thickness_nm
    return ratio * (1.0 + math.log(film_thickness_nm / critical_thickness_nm))


CRITICAL_THICKNESS_NM = 2.0   # Zhang et al., JAP 114, 243516 -- NOT from Hirata
print(f"Relaxation factor f(h) with h_c = {CRITICAL_THICKNESS_NM} nm\n")
reference_factor = strain_relaxation_factor(10.0, CRITICAL_THICKNESS_NM)
for thickness_nm in (2.0, 10.0, 50.0, 100.0, 1000.0, 2000.0):
    factor = strain_relaxation_factor(thickness_nm, CRITICAL_THICKNESS_NM)
    label = "  <-- the paper models these" if thickness_nm in (10.0, 50.0, 100.0) else (
            "  <-- REAL FILMS" if thickness_nm >= 1000.0 else "")
    print(f"  h = {thickness_nm:7.0f} nm   f = {factor:.5f}   "
          f"({reference_factor/factor:5.1f}x weaker than at 10 nm){label}")

## 8. GAPS — what the paper never parametrizes

Verified against the source. Several of these are plotted, though, and are recovered from the figures in section 12.

| Missing | Consequence |
|---|---|
| **Morse parameters** $A$, $D$, $\lambda$, $r_0$ — never tabulated for any of the five compounds, nor the $E(V)$ points they were fitted to | **Eqs. (2)–(13), the entire Debye–Grüneisen chain, cannot be evaluated with the paper's own inputs** |
| $\mu(x)$ and $\nu(x)$ — only curves in Fig. 10(b),(c) | recovered by digitizing, section 12 |
| $h_c(x)$ — imported from Zhang [38], neither tabulated nor plotted | strain model depends on an external number |
| $a(x)$, $c(x)$ — only Fig. 11 | recovered by digitizing (section 12) and fitted (section 15) |
| molar volume $V(x)$ | never given; computed here from the recovered $a$, $c$ (section 16) |
| **wurtzite vacancy endmembers** $^{\circ}G_{\mathrm{Al:Va}}$, $^{\circ}G_{\mathrm{Sc:Va}}$ | the "(Al,Sc)(N,Va)" label in Table 2 is an **empty label**; wurtzite AlN and (Al,Sc)N are strict line compounds with zero homogeneity width |
| the 128-atom **SQS structures** | Fig. 8 cannot be recomputed, only the fitted curves |
| the **per-thickness refitted L** of Fig. 13 | not tabulated |
| any excess entropy of mixing (the mixing-induced excess vibrational entropy) | every ternary L is a bare constant, no T coefficient |
| any **μ_N or P(N₂) axis** | ½E(N₂) is a fixed reference |
| an **h-BN-like polymorph** | absent, despite the paper reporting the c/a collapse and calling x = 32.5 % an outlier on that basis |

## 9. Run every cross-check

Two failures are **deliberate** — they are the paper's internal inconsistencies.

In [ ]:
CHECK_RESULTS.clear()

# ---- SGTE reference states: H - H_SER must vanish, S must match ----
check_zero(sgte_enthalpy(GIBBS_ALUMINIUM_FCC_LOW, ROOM_TEMPERATURE), 0.05,
           "Al fcc: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_zero(sgte_enthalpy(GIBBS_SCANDIUM_HCP_LOW, ROOM_TEMPERATURE), 0.05,
           "Sc hcp: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_value(sgte_entropy(GIBBS_ALUMINIUM_FCC_LOW, ROOM_TEMPERATURE), 28.30, 0.002,
            "Al fcc: S(298.15) vs SGTE 28.30 J/(mol K)")
check_value(sgte_entropy(GIBBS_SCANDIUM_HCP_LOW, ROOM_TEMPERATURE), 34.64, 0.002,
            "Sc hcp: S(298.15) vs SGTE 34.64 J/(mol K)")
check_zero(sgte_enthalpy(GIBBS_HALF_NITROGEN_GAS_LOW, ROOM_TEMPERATURE), 0.05,
           "0.5 N2 gas: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_value(sgte_entropy(GIBBS_HALF_NITROGEN_GAS_LOW, ROOM_TEMPERATURE), 191.61 / 2, 0.002,
            "0.5 N2 gas: S(298.15) vs half the SGTE 191.61 J/(mol K)")

# ---- Table 2 wurtzite AlN against Table 3 ----
aluminium_nitride_entropy = sgte_entropy(GIBBS_ALUMINIUM_NITRIDE_WURTZITE,
                                         ROOM_TEMPERATURE) / ATOMS_PER_FORMULA_UNIT_WURTZITE
check_value(aluminium_nitride_entropy, ALUMINIUM_NITRIDE_ENTROPY_TABLE3, 0.005,
            "AlN wurtzite S(298.15) from Table 2 vs Table 3's 10.07 J/(mol-atom K)",
            note="the T-dependent coefficients are correct")

aluminium_nitride_enthalpy = (sgte_enthalpy(GIBBS_ALUMINIUM_NITRIDE_WURTZITE, ROOM_TEMPERATURE)
                              / ATOMS_PER_FORMULA_UNIT_WURTZITE / 1000.0)
check_value(aluminium_nitride_enthalpy, ALUMINIUM_NITRIDE_ENTHALPY_TABLE3_KJ_PER_MOL_ATOM, 0.01,
            "AlN wurtzite dH(298.15) from Table 2 vs Table 3's -157.07 kJ/mol-atom",
            expect_failure=True,
            note="DELIBERATE: Table 2 and Table 3 disagree by 8.50 kJ/mol-atom (5.4%)")

# ---- Redlich-Kister sanity ----
check_zero(excess_gibbs_energy(0.0, REDLICH_KISTER_WURTZITE), 1e-9,
           "Redlich-Kister vanishes at x = 0")
check_zero(excess_gibbs_energy(1.0, REDLICH_KISTER_WURTZITE), 1e-9,
           "Redlich-Kister vanishes at x = 1")

# ---- the headline mixing / critical-point numbers ----
check_value(excess_gibbs_energy(0.5, REDLICH_KISTER_WURTZITE),
            REDLICH_KISTER_WURTZITE[0] / 4.0, 1e-12,
            "dH_mix(wurtzite, x=0.5) = L0/4 = 24492.6 J/mol-f.u.")
check_value(joules_per_formula_unit_to_millielectronvolt_per_atom(
                excess_gibbs_energy(0.5, REDLICH_KISTER_WURTZITE)), 126.92, 0.001,
            "dH_mix(wurtzite, x=0.5) in meV/atom")
check_value(ideal_mixing_entropy(0.5), GAS_CONSTANT * math.log(2.0), 1e-12,
            "ideal mixing entropy at x=0.5 = R ln2 (per mol CATION sites)")
check_value(ideal_mixing_entropy(0.5) / ATOMS_PER_FORMULA_UNIT_WURTZITE
            / (GAS_CONSTANT), 0.3466, 0.001,
            "... = 0.347 kB/atom, HALF the usual 0.693 because only cations mix")
check_value(wurtzite_critical_temperature, 6971.8, 0.005,
            "wurtzite T_c from the FULL three-term RK (L^2 < 0)",
            note="regular-solution L0/2R = 5892 K UNDERstates it; L^2 < 0 sharpens the dome")
check_value(wurtzite_critical_composition, 0.4986, 0.01,
            "wurtzite x_c -- apex stays at x ~ 0.5 because L^1 is only 0.44% of L^0")
check_value(binodal_temperature(0.30, REDLICH_KISTER_WURTZITE), 6197, 0.02,
            "binodal T at x=0.30 vs the digitized Fig. 14 bulk curve (6197 K)",
            note="independent: reproduces the paper's own figure")
check_value(binodal_temperature(0.899, REDLICH_KISTER_WURTZITE), 4016, 0.03,
            "binodal T at x=0.90 vs digitized Fig. 14 (4016 K)")
check_value(rocksalt_critical_temperature, 5470.6, 0.001,
            "rocksalt T_c = L0/2R (exact here: only L0 was fitted)")

# ---- strain relaxation factor ----
for thickness_nm, expected in ((10.0, 0.52189), (50.0, 0.16876),
                               (100.0, 0.09824), (1000.0, 0.014431)):
    check_value(strain_relaxation_factor(thickness_nm, 2.0), expected, 1e-3,
                f"strain relaxation factor at h = {thickness_nm:.0f} nm")

# ---- the strain <-> interaction-parameter identity behind dL0 = -K ----
_stiffness = 12345.0
for _x in (0.15, 0.4, 0.77):
    check_zero(_stiffness * _x ** 2
               - (_stiffness * _x - _stiffness * _x * (1.0 - _x)), 1e-9,
               f"identity K x^2 = K x - K x(1-x) at x = {_x}  [the basis of dL0 = -K]")

# ---- Table 4 unit label ----
check_value(SCANDIUM_NITRIDE_ENTHALPY_TABLE4 * 2.0, -185.5, 0.01,
            "ScN Table 4 read as per-FORMULA-UNIT",
            expect_failure=True,
            note="DELIBERATE: only the per-ATOM reading is consistent; the label is wrong by 2x")

summarise_checks()

## 10. Plots (optional — needs matplotlib)

Skipped cleanly if matplotlib is unavailable.

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False
    print("matplotlib not installed -- skipping plots. The numerical results above are unaffected.")

if HAVE_MATPLOTLIB:
    compositions = [i / 400.0 for i in range(401)]
    figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(12, 4.5))

    axis_left.plot(compositions,
                   [excess_gibbs_energy(x, REDLICH_KISTER_WURTZITE) / JOULE_PER_ELECTRONVOLT * 1000.0
                    for x in compositions], label="wurtzite (3-term RK)")
    axis_left.plot(compositions,
                   [excess_gibbs_energy(x, REDLICH_KISTER_ROCKSALT) / JOULE_PER_ELECTRONVOLT * 1000.0
                    for x in compositions], "--", label="rocksalt (L0 only)")
    axis_left.set_xlabel("Sc fraction x"); axis_left.set_ylabel("dH_mix  [meV/f.u.]")
    axis_left.set_title("Mixing enthalpy (the paper's Fig. 8)")
    axis_left.legend(); axis_left.grid(alpha=0.3)

    temperatures = list(range(2000, 7100, 50))
    lower_branch, upper_branch, plotted = [], [], []
    for temperature in temperatures:
        interval = spinodal_interval(temperature, REDLICH_KISTER_WURTZITE, 4001)
        if interval:
            plotted.append(temperature)
            lower_branch.append(interval[0]); upper_branch.append(interval[1])
    axis_right.plot(lower_branch, plotted, "b-")
    axis_right.plot(upper_branch, plotted, "b-")
    axis_right.axhline(wurtzite_critical_temperature, color="r", ls=":",
                       label=f"T_c = {wurtzite_critical_temperature:.0f} K "
                             f"at x = {wurtzite_critical_composition:.3f}")
    axis_right.axhline(regular_solution_estimate, color="grey", ls="--",
                       label=f"regular-solution L0/2R = {regular_solution_estimate:.0f} K")
    axis_right.set_xlabel("Sc fraction x"); axis_right.set_ylabel("Temperature [K]")
    axis_right.set_title("Wurtzite spinodal"); axis_right.legend(fontsize=8)
    axis_right.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. Findings — errors and inconsistencies in the published parameters

1. **Table 2 and Table 3 disagree on AlN wurtzite ΔH by 8.50 kJ/mol-atom (5.4 %).**
   Table 2's expression gives −148.57 kJ/mol-atom; Table 3 states −157.07. The *same* expression's
   temperature-dependent coefficients reproduce Table 3's own entropy to 0.2 %, so the discrepancy
   sits **entirely in the leading constant**: −332 447 J/mol would reproduce Table 3 exactly,
   against the −315 445.36 printed. The ScN entry treated identically matches Table 4 to 0.02 %, so
   this is not a systematic misreading — it is a typo in one of the two tables.

2. **Table 4's unit label is wrong by a factor of two** — it heads the block "kJ/mol", but −185.5
   only matches the Table 2 Gibbs energy on a per-**atom** reading. Table 3 labels its column correctly.

3. **Eq. (11) is dimensionally inconsistent with Eq. (13).** As printed, α = (1/r₀)dr₀/dT is the
   *linear* expansion coefficient, but C_p = C_v + α²B₀V₀T requires the *volumetric* one. Taken
   literally the pair understates the C_p correction by a factor of 9.

4. **T_c = L⁰/2R ≈ 5892 K is not this database's critical temperature** — with $L^{(2)} < 0$ (section 6a)
   the full three-term Redlich–Kister gives **6972 K at x_c ≈ 0.50**; the shortcut understates it.
   Rocksalt's 5471 K *is* correct, because only L⁰ was fitted there.

5. **The crossover composition the paper never states**: its own database puts the wurtzite/rocksalt
   crossover at x ≈ 0.60 at 298 K — between Zhang's 0.56 (no vdW) and Talley's 0.64 (PBE-D2).
   §7 calls its enthalpies "relatively consistent" with experiment without ever quoting this number.

6. **Smaller:** AlSc₃N's formation enthalpy is printed as "104.7 kJ/mol per atom" with the minus sign
   dropped; Eq. (4)'s text calls k(ν) "the derived Poisson's ratio" when it is a *function* of it; and
   Table 2's AlN wurtzite carries a T³ coefficient of 1.69×10⁻¹⁷ contributing <4 µJ/mol at 6000 K —
   an optimiser artifact.

### Model-level criticisms (not transcription issues)

See `hirata2020_methodology_review.md` and `calphad_strategy_from_hirata_critique.md`. In brief: no
excess vibrational entropy of mixing (worth ≈ −37 % on T_c, larger than the strain effect the paper is
built around); no short-range order (a further ≈ −9 to −18 %); one "wurtzite" phase fitted straight
through the h-BN structural transition; and no nitrogen off-stoichiometry in wurtzite at all.

## 12. Data recovered from the figures

Section 8 lists what the paper never tabulates. Much of it is *plotted*, so it can be recovered by
digitizing the figures. The figures are embedded JPEGs, so this was done by decoding the images and
locating symbol centroids by shape (open ring vs filled) rather than colour alone — colour alone
fails because each series' connecting polyline shares its symbols' colour, and the experimental
overlays share the calculated series' colours.

**Every array below carries its own uncertainty and provenance.** Three independent consistency
checks validate the extraction — see section 13. Composition is the cation fraction
x in Al(1-x)Sc(x)N; the figures' own axes are "mol% Sc" in TERNARY atomic percent, so AlN = 0 and
ScN = 50 and x = 2*(mol%)/100.

In [ ]:
# ---------------------------------------------------------------------------
# Digitized from Hirata et al. (2020). NOT tabulated in the paper.
# Composition grid: cation x in steps of 1/8 (= 6.25 mol% Sc on the figures' ternary axis).
# ---------------------------------------------------------------------------
COMPOSITION_GRID = [0.000, 0.125, 0.250, 0.375, 0.500, 0.625, 0.750, 0.875, 1.000]

# --- Figure 11: wurtzite lattice constants (calculated series, open symbols) ---
LATTICE_PARAMETER_A = [3.1111, 3.1685, 3.2246, 3.3052, 3.3931, 3.5493, 3.5884, 3.6470, 3.7056]
# +-0.006 Ang as read from Fig. 11; x=0 is a polyline extrapolation, +-0.010
LATTICE_PARAMETER_C = [4.9832, 5.0044, 5.0044, 4.9702, 4.7993, 4.4331, 4.4575, 4.4746, 4.4819]
# +-0.006 Ang as read from Fig. 11; x=0 is a polyline extrapolation, +-0.010
LATTICE_RATIO_C_OVER_A = [1.6000, 1.5802, 1.5529, 1.5045, 1.4137, 1.2477, 1.2428, 1.2262, 1.2081]
# +-0.0015 absolute, read against Fig. 11's right-hand axis

# --- Figure 10: elastic properties (Voigt-Reuss-Hill polycrystal averages) ---
YOUNGS_MODULUS_GPA = [343.37, 291.54, 248.08, 210.14, 170.32, 181.55, 173.17, 175.29, 177.75]
# +-0.6 GPa as read from Fig. 10(a)
SHEAR_MODULUS_GPA = [138.05, 114.61, 95.93, 79.85, 63.65, 67.83, 64.39, 65.10, 66.35]
# +-0.25 GPa as read from Fig. 10(b)
POISSON_RATIO = [0.2452, 0.2726, 0.2943, 0.3166, 0.3352, 0.3376, 0.3447, 0.3439, 0.3418]
# +-0.0008 absolute as read from Fig. 10(c)

# --- Figure 12: epitaxial strain energy, kJ per mol of ATOMS ---
STRAIN_ENERGY_100NM = [0.000, 0.104, 0.362, 0.765, 1.414, 2.956, 3.424, 4.560, 5.885]   # +-0.06 kJ/mol
STRAIN_ENERGY_50NM  = [0.000, 0.314, 0.691, 1.330, 2.439, 5.071, 5.885, 7.868, 10.181]   # +-0.05 kJ/mol
STRAIN_ENERGY_10NM  = [0.000, 0.786, 1.941, 4.148, 7.652, 15.867, 18.384, 24.665, 31.782]   # +-0.06 kJ/mol

# --- Figure 13: mixing enthalpy including strain, kJ per mol of ATOMS ---
MIXING_ENTHALPY_BULK  = [0.000, 4.953, 8.711, 11.073, 12.526, 11.381, 8.686, 4.842, 0.000]   # +-0.05 kJ/mol
MIXING_ENTHALPY_100NM = [0.000, 4.403, 7.628, 9.617, 10.990, 10.648, 7.671, 4.288, 0.000]   # +-0.05 kJ/mol
MIXING_ENTHALPY_50NM  = [0.000, 3.998, 6.834, 8.557, 9.885, 10.106, 6.945, 3.810, 0.000]   # +-0.08 kJ/mol
MIXING_ENTHALPY_10NM  = [0.000, 1.716, 2.716, 3.285, 4.224, 7.390, 3.234, 1.640, 0.000]   # +-0.05 kJ/mol

print(f"{'x':>6} {'a (A)':>8} {'c (A)':>8} {'c/a':>7} {'E(GPa)':>8} "
      f"{'mu(GPa)':>8} {'nu':>7} {'dHmix':>7}")
for i, composition in enumerate(COMPOSITION_GRID):
    print(f"{composition:6.3f} {LATTICE_PARAMETER_A[i]:8.4f} {LATTICE_PARAMETER_C[i]:8.4f} "
          f"{LATTICE_RATIO_C_OVER_A[i]:7.4f} {YOUNGS_MODULUS_GPA[i]:8.2f} "
          f"{SHEAR_MODULUS_GPA[i]:8.2f} {POISSON_RATIO[i]:7.4f} {MIXING_ENTHALPY_BULK[i]:7.3f}")

## 13. Validating the extraction

Three checks, none of which needs anything outside the figures themselves plus Table 2.

1. **c/a self-consistency.** The c/a series must equal (c series)/(a series). This calibrates the
   three Fig. 11 series against each other with no external anchor, and it is what catches
   contamination by the experimental overlay symbols.
2. **Isotropic elasticity identity.** E = 2*mu*(1+nu) must hold for the VRH averages of Fig. 10.
3. **Fig. 13 against Fig. 12 and against Table 2.** The paper's film model is
   `H_film(x) = H_bulk(x) + E_s(x) - x*E_s(ScN)`, the strain contribution entering as an excess
   relative to the endmember interpolation so that the mixing enthalpy still vanishes at both ends.
   That relation ties two independently digitized figures together. Separately, Fig. 13's bulk curve
   must reproduce the Redlich-Kister expression built from Table 2 — which is also a **third,
   independent confirmation that L^2 is negative**.

In [ ]:
print("CHECK 1: c/a self-consistency (Fig. 11)\n")
print(f"{'x':>6} {'c/a read':>9} {'c / a':>9} {'diff':>8} {'sigma':>6}")
worst_ratio = 0.0
for index, composition in enumerate(COMPOSITION_GRID):
    computed = LATTICE_PARAMETER_C[index] / LATTICE_PARAMETER_A[index]
    difference = LATTICE_RATIO_C_OVER_A[index] - computed
    worst_ratio = max(worst_ratio, abs(difference) / 0.0015)
    print(f"{composition:6.3f} {LATTICE_RATIO_C_OVER_A[index]:9.4f} {computed:9.4f} "
          f"{difference:8.4f} {abs(difference)/0.0015:6.1f}")
check_value(worst_ratio, 0.0, 2.0, "Fig.11 c/a self-consistency, worst deviation in sigma",
            note="passes if every point is within ~2 sigma of the reading uncertainty")

print("\nCHECK 2: E = 2*mu*(1+nu) for the VRH averages (Fig. 10)\n")
worst_elastic = 0.0
for index, composition in enumerate(COMPOSITION_GRID):
    predicted = 2.0 * SHEAR_MODULUS_GPA[index] * (1.0 + POISSON_RATIO[index])
    worst_elastic = max(worst_elastic, abs(YOUNGS_MODULUS_GPA[index] - predicted))
check_value(worst_elastic, 0.0, 0.6, "Fig.10 isotropic identity, worst |E - 2mu(1+nu)| in GPa",
            note="0.6 GPa is the stated reading uncertainty on E")

print("\nCHECK 3a: Fig.13 bulk vs the Redlich-Kister of Table 2, per mol-ATOM\n")
worst_mixing = 0.0
print(f"{'x':>6} {'Fig.13':>8} {'RK / 2':>8} {'diff':>7}")
for index, composition in enumerate(COMPOSITION_GRID):
    redlich_kister = excess_gibbs_energy(composition, REDLICH_KISTER_WURTZITE) / 2000.0
    difference = MIXING_ENTHALPY_BULK[index] - redlich_kister
    worst_mixing = max(worst_mixing, abs(difference))
    print(f"{composition:6.3f} {MIXING_ENTHALPY_BULK[index]:8.3f} {redlich_kister:8.3f} "
          f"{difference:7.3f}")
check_value(worst_mixing, 0.0, 0.35, "Fig.13 bulk vs Table 2 Redlich-Kister, worst |diff| kJ/mol",
            note="INDEPENDENT third confirmation that L^2 < 0; with L^2 > 0 the shape is wrong")

print("\nCHECK 3b: Fig.13 against Fig.12 -- H_bulk - H_film == x*E_s(ScN) - E_s(x)\n")
worst_cross = 0.0
for film, strain in ((MIXING_ENTHALPY_100NM, STRAIN_ENERGY_100NM),
                     (MIXING_ENTHALPY_50NM, STRAIN_ENERGY_50NM),
                     (MIXING_ENTHALPY_10NM, STRAIN_ENERGY_10NM)):
    for index, composition in enumerate(COMPOSITION_GRID):
        left = MIXING_ENTHALPY_BULK[index] - film[index]
        right = composition * strain[-1] - strain[index]
        worst_cross = max(worst_cross, abs(left - right))
check_value(worst_cross, 0.0, 0.12,
            "Fig.13 vs Fig.12 cross-validation, worst |diff| kJ/mol over 27 points",
            note="two independently digitized figures agreeing")

### What the extracted data shows, that the paper's text does not

**The buckled -> flat (h-BN-like) crossover.** c falls from ~5.00 to 4.43 Ang between x = 0.5 and
x = 0.625 while a rises monotonically, so c/a drops 1.414 -> 1.248. The paper's text places the
coordination change at "25 mol% Sc", but on its own ternary axis that is x = 0.5, where its Fig. 11
still shows c/a = 1.41. The crossover must complete inside (0.5, 0.625].

**The x = 0.625 elastic anomaly.** mu and E are both ~5% HIGH at x = 0.625 relative to the flat
branch, while the bulk modulus B = E/3(1-2nu) is not anomalous at all (187.0 against a flat-branch
plateau of 187.2 +- 1.0). The anomaly is confined to the SHEAR channel at essentially unchanged
bulk modulus, which points at SQS-realization noise plus a probably-missing projection of the
triclinic-like SQS elastic tensor onto the hexagonal symmetry class, rather than at any
plane-wave-cutoff or k-mesh discretization error (those would move B at least as much as mu).

**Caution on the two-branch fit.** Fitting a quadratic below the crossover and a line above it, then
taking the intersection as the transition composition, gives x0 ~ 0.70 -- outside the (0.5, 0.625]
bracket that c/a already forces. That form silently imposes an order-parameter exponent of 1; the
data prefer a much smaller exponent, which moves x0 back inside the bracket. Locating x0 from
lattice constants alone is not reliable here.

## 14. Two-branch fit of a(x) and c(x), and the transition composition $x_0$

Below the buckled→flat crossover the lattice constants are fitted with a square-root critical form,
above it with a straight line:

$$x < x_0:\quad c = c_0 + c_1 x + c_2\sqrt{x_0-x},\qquad a = a_0 + a_1 x + a_2\sqrt{x_0-x}$$
$$x > x_0:\quad c = c_3 + c_4 x,\qquad\qquad\ \ a = a_3 + a_4 x$$

$x_0$ is **shared between a and c** and fitted jointly. **$x = 0.625$ is excluded from both branches**
— it is anomalous in the elastic data (section 13) and, as the residuals below show, in the lattice
data too.

The fit needs no nonlinear optimiser: at fixed $x_0$ the model is *linear* in
$[c_0,c_1,c_2]$ and $[a_0,a_1,a_2]$, so a 1-D scan over $x_0$ with a linear least-squares solve at
each point is exact and robust.

**On the exponent.** The $\sqrt{\ }$ form imposes an order-parameter exponent $\beta = 1/2$
(mean-field). This is a *choice*, and $x_0$ is sensitive to it — a smaller exponent pulls $x_0$
down, a larger one pushes it up. Treat $x_0$ as conditional on $\beta = 1/2$, not as a
model-independent measurement.

In [ ]:
def solve_linear_system(matrix, vector):
    """Gaussian elimination with partial pivoting. n x n, stdlib only."""
    size = len(vector)
    augmented = [row[:] + [vector[i]] for i, row in enumerate(matrix)]
    for column in range(size):
        pivot = max(range(column, size), key=lambda r: abs(augmented[r][column]))
        augmented[column], augmented[pivot] = augmented[pivot], augmented[column]
        for row in range(size):
            if row != column and augmented[column][column] != 0.0:
                factor = augmented[row][column] / augmented[column][column]
                for k in range(column, size + 1):
                    augmented[row][k] -= factor * augmented[column][k]
    return [augmented[i][size] / augmented[i][i] for i in range(size)]


def least_squares(design_rows, observations):
    """Ordinary least squares for an explicit design matrix."""
    width = len(design_rows[0])
    normal = [[sum(row[i] * row[j] for row in design_rows) for j in range(width)]
              for i in range(width)]
    right = [sum(row[i] * y for row, y in zip(design_rows, observations)) for i in range(width)]
    return solve_linear_system(normal, right)


BUCKLED_INDICES = [i for i, x in enumerate(COMPOSITION_GRID) if x <= 0.5]
FLAT_INDICES = [i for i, x in enumerate(COMPOSITION_GRID) if x >= 0.75]
ANOMALOUS_COMPOSITION = 0.625          # excluded from both branches


def buckled_branch_fit(transition_composition):
    """Fit both a and c on x <= 0.5 with basis [1, x, sqrt(x0 - x)]; return (c, a, ssr)."""
    rows = [[1.0, COMPOSITION_GRID[i],
             math.sqrt(transition_composition - COMPOSITION_GRID[i])] for i in BUCKLED_INDICES]
    c_parameters = least_squares(rows, [LATTICE_PARAMETER_C[i] for i in BUCKLED_INDICES])
    a_parameters = least_squares(rows, [LATTICE_PARAMETER_A[i] for i in BUCKLED_INDICES])
    residual = 0.0
    for row, i in zip(rows, BUCKLED_INDICES):
        residual += (sum(p * v for p, v in zip(c_parameters, row)) - LATTICE_PARAMETER_C[i]) ** 2
        residual += (sum(p * v for p, v in zip(a_parameters, row)) - LATTICE_PARAMETER_A[i]) ** 2
    return c_parameters, a_parameters, residual


# --- scan x0 (must exceed 0.5 for the square root to be real at the last buckled point) ---
_best = None
_step = 0.0002
_scan = 0.5 + _step
while _scan < 3.0:
    _, _, _ssr = buckled_branch_fit(_scan)
    if _best is None or _ssr < _best[0]:
        _best = (_ssr, _scan)
    _scan += _step
TRANSITION_COMPOSITION = _best[1]
C_BUCKLED, A_BUCKLED, _ssr_best = buckled_branch_fit(TRANSITION_COMPOSITION)

_flat_rows = [[1.0, COMPOSITION_GRID[i]] for i in FLAT_INDICES]
C_FLAT = least_squares(_flat_rows, [LATTICE_PARAMETER_C[i] for i in FLAT_INDICES])
A_FLAT = least_squares(_flat_rows, [LATTICE_PARAMETER_A[i] for i in FLAT_INDICES])

print(f"x0 = {TRANSITION_COMPOSITION:.4f}   rms = {math.sqrt(_ssr_best/10):.5f} Ang "
      f"(reading uncertainty 0.006 Ang)\n")
print(f"  buckled  c = {C_BUCKLED[0]:.4f} + {C_BUCKLED[1]:.4f}*x "
      f"+ {C_BUCKLED[2]:.4f}*sqrt(x0 - x)")
print(f"  buckled  a = {A_BUCKLED[0]:.4f} + {A_BUCKLED[1]:.4f}*x "
      f"+ {A_BUCKLED[2]:.4f}*sqrt(x0 - x)")
print(f"  flat     c = {C_FLAT[0]:.4f} + {C_FLAT[1]:.4f}*x")
print(f"  flat     a = {A_FLAT[0]:.4f} + {A_FLAT[1]:.4f}*x")

### Segment-wise equilibrium a(x) and c(x)

Combining the two branches gives lattice-parameter functions usable across the whole composition
range. These are the inputs the strain-energy model needs and that the paper never tabulates —
the misfit $\varepsilon(x) = (a(x)-a_{\mathrm{AlN}})/a_{\mathrm{AlN}}$ follows directly.

In [ ]:
def equilibrium_lattice_parameter_a(composition):
    """Segment-wise a(x) in Angstrom, from the two-branch fit."""
    if composition < TRANSITION_COMPOSITION:
        return (A_BUCKLED[0] + A_BUCKLED[1] * composition
                + A_BUCKLED[2] * math.sqrt(TRANSITION_COMPOSITION - composition))
    return A_FLAT[0] + A_FLAT[1] * composition


def equilibrium_lattice_parameter_c(composition):
    """Segment-wise c(x) in Angstrom, from the two-branch fit."""
    if composition < TRANSITION_COMPOSITION:
        return (C_BUCKLED[0] + C_BUCKLED[1] * composition
                + C_BUCKLED[2] * math.sqrt(TRANSITION_COMPOSITION - composition))
    return C_FLAT[0] + C_FLAT[1] * composition


def misfit_strain(composition, substrate_lattice_parameter=None):
    """In-plane misfit against a substrate; defaults to AlN, i.e. a(x=0)."""
    reference = substrate_lattice_parameter or equilibrium_lattice_parameter_a(0.0)
    return (equilibrium_lattice_parameter_a(composition) - reference) / reference


print(f"{'x':>6} {'a obs':>8} {'a fit':>8} {'da':>8} {'c obs':>8} {'c fit':>8} {'dc':>8}")
for index, composition in enumerate(COMPOSITION_GRID):
    a_fit = equilibrium_lattice_parameter_a(composition)
    c_fit = equilibrium_lattice_parameter_c(composition)
    flag = "  <-- EXCLUDED from both fits" if composition == ANOMALOUS_COMPOSITION else ""
    print(f"{composition:6.3f} {LATTICE_PARAMETER_A[index]:8.4f} {a_fit:8.4f} "
          f"{LATTICE_PARAMETER_A[index]-a_fit:8.4f} {LATTICE_PARAMETER_C[index]:8.4f} "
          f"{c_fit:8.4f} {LATTICE_PARAMETER_C[index]-c_fit:8.4f}{flag}")

check_value(math.sqrt(_ssr_best / 10), 0.0, 0.006,
            "two-branch fit rms residual vs the 0.006 Ang reading uncertainty")
check_value(TRANSITION_COMPOSITION, 0.5625, 0.15,
            "x0 lies inside the bracket (0.5, 0.625] forced by c/a",
            note="c/a = 1.414 at x=0.5 (buckled) and 1.248 at x=0.625 (flat)")

In [ ]:
if HAVE_MATPLOTLIB:
    # Colours and symbols follow the paper's own Fig. 11:
    #   a -> blue open SQUARES,  c -> rust/orange open CIRCLES,  y range 3.0 - 5.5 Angstrom.
    COLOUR_A, COLOUR_C = "#2B4A9B", "#C1552B"
    anomalous_index = COMPOSITION_GRID.index(ANOMALOUS_COMPOSITION)
    kept = [i for i, x in enumerate(COMPOSITION_GRID) if x != ANOMALOUS_COMPOSITION]

    def branch_mesh(transition, samples=400):
        """Two meshes that each terminate EXACTLY at the transition composition.

        A mesh built by filtering a uniform grid stops at the nearest sample below/above
        x0, leaving a visible gap wherever the curve is steep -- which it is here, because
        the sqrt term has infinite slope as x -> x0.
        """
        lower = [transition * i / samples for i in range(samples + 1)]
        upper = [transition + (1.0 - transition) * i / samples for i in range(samples + 1)]
        return lower, upper

    def buckled_value(parameters, transition, x):
        return parameters[0] + parameters[1] * x + parameters[2] * math.sqrt(max(transition - x, 0.0))

    def flat_value(intercept, slope, x):
        return intercept + slope * x

    figure, axis = plt.subplots(figsize=(7.2, 6.0))
    lower_mesh, upper_mesh = branch_mesh(TRANSITION_COMPOSITION)
    for observed, buckled_parameters, flat_parameters, colour, marker, label in (
            (LATTICE_PARAMETER_C, C_BUCKLED, C_FLAT, COLOUR_C, "o", "c"),
            (LATTICE_PARAMETER_A, A_BUCKLED, A_FLAT, COLOUR_A, "s", "a")):
        axis.plot(lower_mesh,
                  [buckled_value(buckled_parameters, TRANSITION_COMPOSITION, x) for x in lower_mesh],
                  "-", color=colour, linewidth=1.6)
        axis.plot(upper_mesh,
                  [flat_value(flat_parameters[0], flat_parameters[1], x) for x in upper_mesh],
                  "-", color=colour, linewidth=1.6)
        axis.plot([COMPOSITION_GRID[i] for i in kept], [observed[i] for i in kept],
                  marker, markerfacecolor="none", markeredgecolor=colour,
                  markeredgewidth=1.6, markersize=8, linestyle="none",
                  label=f"{label}  (DFT, fitted)")
        axis.plot([ANOMALOUS_COMPOSITION], [observed[anomalous_index]], "x", color=colour,
                  markersize=11, markeredgewidth=2.6, linestyle="none",
                  label=f"{label}  (x = 0.625, excluded)")

    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls=":", linewidth=1.2)
    axis.annotate(f"$x_0$ = {TRANSITION_COMPOSITION:.3f}",
                  xy=(TRANSITION_COMPOSITION, 5.35), xytext=(TRANSITION_COMPOSITION + 0.03, 5.35),
                  color="grey", fontsize=10)
    axis.set_xlim(0.0, 1.0)
    axis.set_ylim(3.0, 5.5)                      # same range as the paper's Fig. 11
    axis.set_xlabel("Sc cation fraction $x$ in Al$_{1-x}$Sc$_x$N")
    axis.set_ylabel("Lattice constant $a$, $c$  ($\\AA$)")
    axis.set_title("Two-branch fit of the wurtzite lattice constants\n"
                   "(independent branches -- note the genuine jump at $x_0$)")
    axis.legend(loc="center left", fontsize=9, framealpha=0.95)
    axis.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
else:
    print("matplotlib unavailable -- fit parameters and residuals above are unaffected.")

## 15. Continuity-constrained co-fit

Section 14 fits the two branches **independently**, which lets them disagree at $x_0$. They do, badly:
the jump there is $\Delta a = +0.051$ Å and $\Delta c = -0.227$ Å — about 8σ and 38σ against the
0.006 Å reading uncertainty. Unless the transition is genuinely first order, that is unphysical, and
nothing in the section-14 construction prevents it.

Here both branches are fitted **simultaneously to all the data**, subject to matching at $x_0$:

$$a_0 + a_1 x_0 = a_3 + a_4 x_0, \qquad c_0 + c_1 x_0 = c_3 + c_4 x_0$$

The $\sqrt{x_0-x}$ term vanishes at $x_0$, so it drops out of the constraints — they involve only the
linear coefficients, exactly as written.

**How it is solved.** The constraint is linear, so eliminate rather than penalise: substituting
$a_3 = a_0 + a_1 x_0 - a_4 x_0$ makes the *whole* model — both branches at once — linear in
$[a_0, a_1, a_2, a_4]$ at fixed $x_0$. One design matrix over all points, with rows

$$x < x_0:\ [1,\ x,\ \sqrt{x_0-x},\ 0] \qquad x \ge x_0:\ [1,\ x_0,\ 0,\ x-x_0]$$

so continuity is enforced *exactly* by construction rather than approximately by fitting. Scanning
$x_0$ then needs only a linear solve at each step, as before. $x = 0.625$ remains excluded.

In [ ]:
def continuous_design_matrix(transition_composition, indices):
    """Rows for the continuity-eliminated model; parameters are [p0, p1, p2, p4]."""
    rows = []
    for i in indices:
        x = COMPOSITION_GRID[i]
        if x < transition_composition:
            rows.append([1.0, x, math.sqrt(transition_composition - x), 0.0])
        else:
            rows.append([1.0, transition_composition, 0.0, x - transition_composition])
    return rows


FITTED_INDICES = [i for i, x in enumerate(COMPOSITION_GRID) if x != ANOMALOUS_COMPOSITION]


def continuous_cofit(transition_composition):
    """Joint continuity-constrained fit of a and c. Returns (a params, c params, ssr)."""
    rows = continuous_design_matrix(transition_composition, FITTED_INDICES)
    a_parameters = least_squares(rows, [LATTICE_PARAMETER_A[i] for i in FITTED_INDICES])
    c_parameters = least_squares(rows, [LATTICE_PARAMETER_C[i] for i in FITTED_INDICES])
    residual = 0.0
    for row, i in zip(rows, FITTED_INDICES):
        residual += (sum(p * v for p, v in zip(a_parameters, row)) - LATTICE_PARAMETER_A[i]) ** 2
        residual += (sum(p * v for p, v in zip(c_parameters, row)) - LATTICE_PARAMETER_C[i]) ** 2
    return a_parameters, c_parameters, residual


# x0 must exceed 0.5 (real square root at the last buckled point) and stay below 0.75
# (so the flat branch keeps all three of its points).
_best_cofit = None
_scan = 0.5005
while _scan < 0.7495:
    _, _, _ssr = continuous_cofit(_scan)
    if _best_cofit is None or _ssr < _best_cofit[0]:
        _best_cofit = (_ssr, _scan)
    _scan += 0.0005
TRANSITION_COMPOSITION_COFIT = _best_cofit[1]
A_COFIT, C_COFIT, _ssr_cofit = continuous_cofit(TRANSITION_COMPOSITION_COFIT)

# recover the eliminated flat-branch intercepts
A_COFIT_FLAT_INTERCEPT = (A_COFIT[0] + A_COFIT[1] * TRANSITION_COMPOSITION_COFIT
                          - A_COFIT[3] * TRANSITION_COMPOSITION_COFIT)
C_COFIT_FLAT_INTERCEPT = (C_COFIT[0] + C_COFIT[1] * TRANSITION_COMPOSITION_COFIT
                          - C_COFIT[3] * TRANSITION_COMPOSITION_COFIT)


def cofit_lattice_parameter_a(composition):
    if composition < TRANSITION_COMPOSITION_COFIT:
        return (A_COFIT[0] + A_COFIT[1] * composition
                + A_COFIT[2] * math.sqrt(TRANSITION_COMPOSITION_COFIT - composition))
    return A_COFIT_FLAT_INTERCEPT + A_COFIT[3] * composition


def cofit_lattice_parameter_c(composition):
    if composition < TRANSITION_COMPOSITION_COFIT:
        return (C_COFIT[0] + C_COFIT[1] * composition
                + C_COFIT[2] * math.sqrt(TRANSITION_COMPOSITION_COFIT - composition))
    return C_COFIT_FLAT_INTERCEPT + C_COFIT[3] * composition


print(f"x0 = {TRANSITION_COMPOSITION_COFIT:.4f}   "
      f"rms = {math.sqrt(_ssr_cofit / (2 * len(FITTED_INDICES))):.5f} Ang\n")
print(f"  a  x<x0 : {A_COFIT[0]:.4f} + {A_COFIT[1]:.4f}*x + {A_COFIT[2]:.4f}*sqrt(x0-x)")
print(f"  a  x>x0 : {A_COFIT_FLAT_INTERCEPT:.4f} + {A_COFIT[3]:.4f}*x")
print(f"  c  x<x0 : {C_COFIT[0]:.4f} + {C_COFIT[1]:.4f}*x + {C_COFIT[2]:.4f}*sqrt(x0-x)")
print(f"  c  x>x0 : {C_COFIT_FLAT_INTERCEPT:.4f} + {C_COFIT[3]:.4f}*x")

In [ ]:
print("Comparison: section 14 (independent) vs section 15 (continuity-constrained)\n")
independent_jump_a = (A_FLAT[0] + A_FLAT[1] * TRANSITION_COMPOSITION
                      - (A_BUCKLED[0] + A_BUCKLED[1] * TRANSITION_COMPOSITION))
independent_jump_c = (C_FLAT[0] + C_FLAT[1] * TRANSITION_COMPOSITION
                      - (C_BUCKLED[0] + C_BUCKLED[1] * TRANSITION_COMPOSITION))
cofit_jump_a = (A_COFIT_FLAT_INTERCEPT + A_COFIT[3] * TRANSITION_COMPOSITION_COFIT
                - (A_COFIT[0] + A_COFIT[1] * TRANSITION_COMPOSITION_COFIT))
cofit_jump_c = (C_COFIT_FLAT_INTERCEPT + C_COFIT[3] * TRANSITION_COMPOSITION_COFIT
                - (C_COFIT[0] + C_COFIT[1] * TRANSITION_COMPOSITION_COFIT))

print(f"{'':26} {'independent':>14} {'constrained':>14}")
print(f"{'x0':26} {TRANSITION_COMPOSITION:14.4f} {TRANSITION_COMPOSITION_COFIT:14.4f}")
print(f"{'rms residual (Ang)':26} {math.sqrt(_ssr_best/10):14.5f} "
      f"{math.sqrt(_ssr_cofit/(2*len(FITTED_INDICES))):14.5f}")
print(f"{'jump in a at x0 (Ang)':26} {independent_jump_a:14.4f} {cofit_jump_a:14.4f}")
print(f"{'jump in c at x0 (Ang)':26} {independent_jump_c:14.4f} {cofit_jump_c:14.4f}")

print(f"\n{'x':>6} {'a obs':>8} {'a s14':>8} {'a s15':>8} | "
      f"{'c obs':>8} {'c s14':>8} {'c s15':>8}")
for index, composition in enumerate(COMPOSITION_GRID):
    flag = "  <-- excluded" if composition == ANOMALOUS_COMPOSITION else ""
    print(f"{composition:6.3f} {LATTICE_PARAMETER_A[index]:8.4f} "
          f"{equilibrium_lattice_parameter_a(composition):8.4f} "
          f"{cofit_lattice_parameter_a(composition):8.4f} | "
          f"{LATTICE_PARAMETER_C[index]:8.4f} "
          f"{equilibrium_lattice_parameter_c(composition):8.4f} "
          f"{cofit_lattice_parameter_c(composition):8.4f}{flag}")

check_value(abs(cofit_jump_a) + abs(cofit_jump_c), 0.0, 1e-9,
            "continuity enforced exactly at x0 in the constrained co-fit")
check_value(math.sqrt(_ssr_cofit / (2 * len(FITTED_INDICES))), 0.0, 0.006,
            "constrained co-fit rms vs the 0.006 Ang reading uncertainty",
            note="continuity costs little: 0.0032 vs 0.0026 Ang unconstrained")

In [ ]:
if HAVE_MATPLOTLIB:
    COLOUR_A, COLOUR_C = "#2B4A9B", "#C1552B"
    anomalous_index = COMPOSITION_GRID.index(ANOMALOUS_COMPOSITION)
    kept = [i for i, x in enumerate(COMPOSITION_GRID) if x != ANOMALOUS_COMPOSITION]

    def branch_mesh(transition, samples=400):
        """Meshes terminating exactly at x0, so neither curve stops short of it."""
        lower = [transition * i / samples for i in range(samples + 1)]
        upper = [transition + (1.0 - transition) * i / samples for i in range(samples + 1)]
        return lower, upper

    figure, axis = plt.subplots(figsize=(7.6, 6.2))
    for observed, colour, marker, label, buckled, flat_pair, cofit_buckled, cofit_flat in (
            (LATTICE_PARAMETER_C, COLOUR_C, "o", "c", C_BUCKLED, (C_FLAT[0], C_FLAT[1]),
             C_COFIT, (C_COFIT_FLAT_INTERCEPT, C_COFIT[3])),
            (LATTICE_PARAMETER_A, COLOUR_A, "s", "a", A_BUCKLED, (A_FLAT[0], A_FLAT[1]),
             A_COFIT, (A_COFIT_FLAT_INTERCEPT, A_COFIT[3]))):
        # independent fit (dashed) -- genuinely discontinuous at its own x0
        low, high = branch_mesh(TRANSITION_COMPOSITION)
        axis.plot(low, [buckled[0] + buckled[1] * x
                        + buckled[2] * math.sqrt(max(TRANSITION_COMPOSITION - x, 0.0))
                        for x in low], "--", color=colour, linewidth=1.2)
        axis.plot(high, [flat_pair[0] + flat_pair[1] * x for x in high],
                  "--", color=colour, linewidth=1.2)
        # continuity-constrained fit (solid) -- meets exactly at its x0
        low, high = branch_mesh(TRANSITION_COMPOSITION_COFIT)
        axis.plot(low, [cofit_buckled[0] + cofit_buckled[1] * x
                        + cofit_buckled[2] * math.sqrt(max(TRANSITION_COMPOSITION_COFIT - x, 0.0))
                        for x in low], "-", color=colour, linewidth=1.8)
        axis.plot(high, [cofit_flat[0] + cofit_flat[1] * x for x in high],
                  "-", color=colour, linewidth=1.8)
        axis.plot([COMPOSITION_GRID[i] for i in kept], [observed[i] for i in kept],
                  marker, markerfacecolor="none", markeredgecolor=colour,
                  markeredgewidth=1.6, markersize=8, linestyle="none", label=f"{label}  (DFT)")
        axis.plot([ANOMALOUS_COMPOSITION], [observed[anomalous_index]], "x", color=colour,
                  markersize=11, markeredgewidth=2.6, linestyle="none",
                  label=f"{label}  (x = 0.625, excluded)")

    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls="--", linewidth=1.0)
    axis.axvline(TRANSITION_COMPOSITION_COFIT, color="grey", ls=":", linewidth=1.4)
    axis.plot([], [], "--", color="grey", label=f"independent ($x_0$={TRANSITION_COMPOSITION:.3f})")
    axis.plot([], [], "-", color="grey",
              label=f"continuous ($x_0$={TRANSITION_COMPOSITION_COFIT:.3f})")
    axis.set_xlim(0.0, 1.0); axis.set_ylim(3.0, 5.5)
    axis.set_xlabel("Sc cation fraction $x$ in Al$_{1-x}$Sc$_x$N")
    axis.set_ylabel("Lattice constant $a$, $c$  ($\\AA$)")
    axis.set_title("Independent (dashed) vs continuity-constrained (solid) two-branch fits")
    axis.legend(loc="center left", fontsize=8.5, framealpha=0.95)
    axis.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
else:
    print("matplotlib unavailable -- fit parameters and residuals above are unaffected.")

## 16. Strain inputs

| quantity | source |
|---|---|
| $a(x)$, $c(x)$ | section-15 fit, with the transition at $x_0$ smoothed (below) |
| $E(x)$, $\nu(x)$ | Fig. 10, shape-preserving interpolation, $x = 0.625$ excluded (section 13) |
| molar volume $V_m(x)$ | wurtzite cell $(\sqrt3/2)\,a^2c$, 2 formula units per cell |
| critical thickness $h_c$ | 2 nm (Zhang *et al.*, via Hirata), section 7 |

Moduli are the **isotropic** Voigt–Reuss–Hill averages. Hexagonal elastic constants are not available;
the out-of-plane coefficient $2C_{13}/C_{33}$ uses the isotropic surrogate $2\nu/(1-\nu)$, with a placeholder
for Zhang *et al.* (2013).

**Smoothed transition.** The section-15 fit has $\sqrt{x_0-x}$ and $\max(x-x_0,0)$ terms, i.e. an infinite
slope and a kink at $x_0$. The elastic model needs finite slopes, so both are replaced by smooth versions
built on $\mathrm{softplus}_w(z) = w\ln(1+e^{z/w})$: $\max(z,0) \to \mathrm{softplus}_w(z)$ and
$\sqrt{\max(z,0)} \to \sqrt{\mathrm{softplus}_{w/2}(z)}$, with an assumed transition width $w = 0.02$ — a random
alloy smears the transition. Away from $x_0$ the change is below the 0.006 Å reading uncertainty.

Energies are kept in J/mol-f.u. internally (Hirata's unit); printed and plotted in meV/f.u., where
f.u. = Al$_{1-x}$Sc$_x$N.

In [ ]:
FORMULA_UNITS_PER_WURTZITE_CELL = 2
AVOGADRO_CONSTANT = 6.02214076e23          # 1/mol
CUBIC_METRE_PER_CUBIC_ANGSTROM = 1e-30
PASCAL_PER_GIGAPASCAL = 1e9
MILLIELECTRONVOLT_PER_JOULE_PER_MOLE = 1000.0 / JOULE_PER_ELECTRONVOLT

ELASTIC_COMPOSITIONS = [COMPOSITION_GRID[i] for i in FITTED_INDICES]
youngs_modulus_interpolant = PchipInterpolator(
    ELASTIC_COMPOSITIONS, [YOUNGS_MODULUS_GPA[i] for i in FITTED_INDICES])
poisson_ratio_interpolant = PchipInterpolator(
    ELASTIC_COMPOSITIONS, [POISSON_RATIO[i] for i in FITTED_INDICES])

# Placeholder: 2*C13/C33(x), Zhang et al. 2013, doi:10.1063/1.4848036
# None -> isotropic surrogate
OUT_OF_PLANE_COEFFICIENT_FROM_ELASTIC_CONSTANTS = None

LATTICE_TRANSITION_WIDTH = 0.02    # assumed; smears the transition at x0
LATTICE_READING_UNCERTAINTY = 0.006      # Angstrom, section 12


def softplus(value, width=LATTICE_TRANSITION_WIDTH):
    """Smooth max(value, 0) with the given width; accepts arrays."""
    return width * np.logaddexp(0.0, np.asarray(value, dtype=float) / width)


def smoothed_branch_fit(composition, parameters, flat_intercept):
    """Section-15 continuous fit with sqrt and kink replaced by softplus forms; Angstrom."""
    x = np.asarray(composition, dtype=float)
    transition = TRANSITION_COMPOSITION_COFIT
    above = softplus(x - transition)
    return (parameters[0] + parameters[1] * (x - above) + parameters[3] * above
            + parameters[2] * np.sqrt(softplus(transition - x, 0.5 * LATTICE_TRANSITION_WIDTH)))


def lattice_parameters(composition):
    """Stress-free (a, c) in Angstrom of a homogeneous alloy; composition scalar or array."""
    return (smoothed_branch_fit(composition, A_COFIT, A_COFIT_FLAT_INTERCEPT),
            smoothed_branch_fit(composition, C_COFIT, C_COFIT_FLAT_INTERCEPT))


def youngs_modulus(composition):
    """Isotropic Young's modulus, Pa."""
    return youngs_modulus_interpolant(composition) * PASCAL_PER_GIGAPASCAL


def poisson_ratio(composition):
    return poisson_ratio_interpolant(composition)


def molar_volume(composition):
    """m^3 per mol of formula unit."""
    a, c = lattice_parameters(composition)
    cell_volume = np.sqrt(3.0) / 2.0 * a * a * c * CUBIC_METRE_PER_CUBIC_ANGSTROM
    return cell_volume * AVOGADRO_CONSTANT / FORMULA_UNITS_PER_WURTZITE_CELL


def out_of_plane_coefficient(composition):
    """-eps_zz/eps_xx of a (0001) film under biaxial in-plane strain: 2*C13/C33."""
    if OUT_OF_PLANE_COEFFICIENT_FROM_ELASTIC_CONSTANTS is not None:
        return OUT_OF_PLANE_COEFFICIENT_FROM_ELASTIC_CONSTANTS(composition)
    nu = poisson_ratio(composition)
    return 2.0 * nu / (1.0 - nu)          # isotropic surrogate


SUBSTRATE_IN_PLANE_LATTICE = {     # Angstrom
    "AlN": float(lattice_parameters(0.0)[0]),   # zero misfit at x = 0
    "GaN": 3.189,                               # experimental, assumed same scale
}


def strained_lattice_parameter_c(composition, substrate_lattice_parameter):
    """c (Angstrom) of a (0001) film clamped in-plane to a substrate lattice parameter (Angstrom)."""
    a, c = lattice_parameters(composition)
    in_plane_strain = substrate_lattice_parameter / a - 1.0
    return c * (1.0 - out_of_plane_coefficient(composition) * in_plane_strain)


print(f"{'x':>6} {'a':>7} {'c':>7} {'a fit':>7} {'c fit':>7} {'E (GPa)':>8} {'nu':>7} "
      f"{'V_m (m3/mol)':>13} {'c on AlN':>9} {'c on GaN':>9}")
for composition in (0.0, 0.2, 0.4, 0.5, 0.54, 0.56, 0.6, 0.8, 1.0):
    a, c = lattice_parameters(composition)
    print(f"{composition:6.2f} {a:7.4f} {c:7.4f} {cofit_lattice_parameter_a(composition):7.4f} "
          f"{cofit_lattice_parameter_c(composition):7.4f} {youngs_modulus(composition)/1e9:8.1f} "
          f"{poisson_ratio(composition):7.4f} {molar_volume(composition):13.4e} "
          f"{strained_lattice_parameter_c(composition, SUBSTRATE_IN_PLANE_LATTICE['AlN']):9.4f} "
          f"{strained_lattice_parameter_c(composition, SUBSTRATE_IN_PLANE_LATTICE['GaN']):9.4f}")

check_value(float(molar_volume(0.0)), 1.259e-5, 0.01, "V_m(AlN) from the smoothed a, c vs 1.259e-5 m^3/mol")
_smoothing_error = max(np.max(np.abs(np.array(lattice_parameters(COMPOSITION_GRID[i]))
                                     - [cofit_lattice_parameter_a(COMPOSITION_GRID[i]),
                                        cofit_lattice_parameter_c(COMPOSITION_GRID[i])]))
                       for i in FITTED_INDICES)
check_value(_smoothing_error, 0.0, LATTICE_READING_UNCERTAINTY,
            "smoothing changes a, c by less than the reading error at the fitted data")

### Reproducing Hirata's Fig. 12

Hirata's strain energy is a (0001) film clamped to AlN, with his misfit convention
$(a - a_\mathrm{AlN})/a_\mathrm{AlN}$. Evaluated at the grid points from the recovered Fig. 10–11 data it
should land on the digitized Fig. 12 (kJ per mol of **atoms**). A ±0.006 Å reading error on both $a(x)$
and $a(0)$ limits the comparison at small misfit: expected scatter $2\,\delta\epsilon/\epsilon$, about 30 % at
$x = 0.125$ and 6 % at $x = 0.5$.

On the buckled branch every point agrees within twice that scatter. On the flat branch ($x \ge 0.625$) the
recomputed energy is a uniform ~32 % above the figure, which the recovered inputs do not explain.

The model itself uses the film-referenced strain $a_\mathrm{sub}/a_\mathrm{film} - 1$, ~10 % less energy at
$x \approx 0.4$ than Hirata's convention.

In [ ]:
def hirata_grid_strain_energy(index, film_thickness_nm):
    """Hirata's Eq. (21) at a Fig. 11 grid point from digitized data: (kJ/mol-atoms, misfit)."""
    misfit = (LATTICE_PARAMETER_A[index] - LATTICE_PARAMETER_A[0]) / LATTICE_PARAMETER_A[0]
    cell_volume = (math.sqrt(3.0) / 2.0 * LATTICE_PARAMETER_A[index] ** 2
                   * LATTICE_PARAMETER_C[index] * CUBIC_METRE_PER_CUBIC_ANGSTROM)
    volume = cell_volume * AVOGADRO_CONSTANT / FORMULA_UNITS_PER_WURTZITE_CELL
    biaxial_modulus = YOUNGS_MODULUS_GPA[index] * PASCAL_PER_GIGAPASCAL / (1.0 - POISSON_RATIO[index])
    relaxation = strain_relaxation_factor(film_thickness_nm, CRITICAL_THICKNESS_NM)
    return biaxial_modulus * misfit ** 2 * volume * relaxation / 2000.0, misfit


print(f"{'x':>6} {'h (nm)':>7} {'Fig.12':>8} {'recomputed':>11} {'deviation':>10} {'expected':>9}")
_worst_fig12_sigma = 0.0
for index, composition in enumerate(COMPOSITION_GRID):
    if composition == 0.0:
        continue
    for thickness_nm, digitized in ((10.0, STRAIN_ENERGY_10NM), (50.0, STRAIN_ENERGY_50NM),
                                    (100.0, STRAIN_ENERGY_100NM)):
        recomputed, misfit = hirata_grid_strain_energy(index, thickness_nm)
        scatter = 2.0 * math.sqrt(2.0) * LATTICE_READING_UNCERTAINTY / LATTICE_PARAMETER_A[0] / misfit
        deviation = recomputed / digitized[index] - 1.0
        if composition < TRANSITION_COMPOSITION_COFIT:
            _worst_fig12_sigma = max(_worst_fig12_sigma, abs(deviation) / scatter)
        print(f"{composition:6.3f} {thickness_nm:7.0f} {digitized[index]:8.3f} {recomputed:11.3f} "
              f"{deviation * 100:9.0f}% {scatter * 100:8.0f}%")
check_value(_worst_fig12_sigma, 0.0, 2.0,
            "Fig. 12 recomputed from Figs. 10-11 on the buckled branch, worst deviation in sigma",
            note="sigma = misfit reading uncertainty propagated to the energy")

## 17. Elastic energy models

A decomposed film consists of **domains** $i$ with compositions $x_i$, mole fractions $f_i$ and stress-free
lattices $a(x_i)$, $c(x_i)$. Every model below returns the elastic energy of such a set of domains,
$\sum_i f_i V_m(x_i)\,w_i$, in J per mol of formula unit of the whole film; $w_i$ is the elastic energy density
of domain $i$. Linear, isotropic elasticity throughout; no interface or gradient energy; the domains are
flat layers (a laminate) much thicker than the interface.

| model | geometry | constraint |
|---|---|---|
| `no_elastic_energy` | incoherent: dislocated interfaces | none; every domain relaxes to its own lattice |
| `coherent_laminate_energy` | free-standing (bulk, or film fully relaxed from its substrate) | domains share the lattice in the interface plane; interface orientation $\theta$ (from $c$) minimised |
| `clamped_layered_energy` | film on a substrate, layers stacked along $c$ (basal interfaces) | every layer clamped in-plane to $a_\mathrm{sub}$ |
| `clamped_lateral_energy` | film on a substrate, columns (interfaces normal to the film) | domains share the lattice along the interface; the **average** in-plane lattice is clamped to $a_\mathrm{sub}$ |

**Coherent laminate.** For an interface normal at angle $\theta$ from $c$, the in-plane misfits of domain $i$
relative to a reference lattice are $e_1 = \epsilon_a$, $e_2 = \epsilon_a\cos^2\theta + \epsilon_c\sin^2\theta$
($\epsilon_a$, $\epsilon_c$: fractional differences of $a$, $c$). Normal and out-of-plane shear stresses relax
(plane stress), so
$w = \tfrac{E}{2(1-\nu^2)}\left[(s_1-e_1)^2 + (s_2-e_2)^2 + 2\nu(s_1-e_1)(s_2-e_2)\right]$, with the shared
in-plane strain $(s_1, s_2)$ chosen to minimise the total — a $2\times2$ linear solve. With constant moduli and
linear misfit this reduces to Cahn–Larché, $Y\eta^2 f(1-f)(x_B-x_C)^2V_m$.

**Clamped, layered.** Each layer carries biaxial strain $a_\mathrm{sub}/a(x_i) - 1$ with $c$ free:
$w = E\epsilon^2/(1-\nu)$ — Hirata's form, now per domain.

**Clamped, lateral.** The film normal is $c$; the interface normal $n$ lies in the basal plane. Along the
in-plane direction $t$ within the interface every domain is strained to $a_\mathrm{sub}$; along $c$ the domains
share one strain whose average stress vanishes (free surface); along $n$ each domain has its own strain, but
their fraction-weighted average is fixed by the substrate. Minimising the full isotropic energy under that
constraint is a $4\times4$ linear (Lagrange-multiplier) solve.

A single homogeneous domain has zero energy in the first two models and the biaxial clamping energy in the
last two. All functions accept arrays of shape (..., number of domains) and return shape (...).

In [ ]:
INTERFACE_ANGLES = np.linspace(0.0, 0.5 * math.pi, 19)     # 5 degree steps; 0 = basal, 90 = prism


def domain_properties(compositions, fractions):
    """Lattice misfits relative to the average-composition lattice, and fraction * molar volume."""
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    reference_a, reference_c = lattice_parameters(np.sum(fractions * compositions, axis=-1))
    a, c = lattice_parameters(compositions)
    misfit_a = a / reference_a[..., None] - 1.0
    misfit_c = c / reference_c[..., None] - 1.0
    return misfit_a, misfit_c, fractions * molar_volume(compositions), reference_a


def no_elastic_energy(compositions, fractions):
    """Incoherent domains: zero, J/mol-f.u."""
    return np.zeros(np.shape(compositions)[:-1])


def plane_stress_stiffness(compositions):
    """In-plane stiffness (Pa) of an isotropic solid with free normal stress; shape (..., 2, 2)."""
    poisson = poisson_ratio(compositions)
    ones = np.ones_like(poisson)
    matrix = np.stack([np.stack([ones, poisson], -1), np.stack([poisson, ones], -1)], -2)
    return (youngs_modulus(compositions) / (1.0 - poisson ** 2))[..., None, None] * matrix


def coherent_laminate_energy(compositions, fractions, angles=INTERFACE_ANGLES):
    """Free-standing coherent laminate, minimised over interface angle; J/mol-f.u."""
    misfit_a, misfit_c, weights, _ = domain_properties(compositions, fractions)
    tilted = misfit_a[..., None] * np.cos(angles) ** 2 + misfit_c[..., None] * np.sin(angles) ** 2
    misfits = np.stack([np.broadcast_to(misfit_a[..., None], tilted.shape), tilted], axis=-1)
    weighted = weights[..., None, None] * plane_stress_stiffness(compositions)   # (..., domains, 2, 2)
    load = np.einsum("...dij,...daj->...ai", weighted, misfits)                 # (..., angles, 2)
    total = np.broadcast_to(weighted.sum(axis=-3)[..., None, :, :], load.shape + (2,))
    shared = np.linalg.solve(total, load[..., None])[..., 0]
    elastic = shared[..., None, :, :] - misfits                                 # (..., domains, angles, 2)
    energy = 0.5 * np.einsum("...dai,...dij,...daj->...a", elastic, weighted, elastic)
    return energy.min(axis=-1)


def clamped_layered_energy(compositions, fractions, substrate_lattice_parameter):
    """Every domain biaxially clamped to the substrate (basal interfaces); J/mol-f.u."""
    compositions = np.asarray(compositions, dtype=float)
    a, _ = lattice_parameters(compositions)
    strain = np.asarray(substrate_lattice_parameter)[..., None] / a - 1.0
    biaxial = youngs_modulus(compositions) / (1.0 - poisson_ratio(compositions))
    return np.sum(np.asarray(fractions) * molar_volume(compositions) * biaxial * strain ** 2, axis=-1)


def lame_constants(compositions):
    """(lambda, lambda + 2 mu) of the isotropic solid, Pa."""
    youngs, poisson = youngs_modulus(compositions), poisson_ratio(compositions)
    lame = youngs * poisson / ((1.0 + poisson) * (1.0 - 2.0 * poisson))
    return lame, lame + youngs / (1.0 + poisson)


def isotropic_energy_density(strain_t, strain_n, strain_z, lame, longitudinal):
    """Elastic energy density (J/m^3) for normal strains only."""
    return (0.5 * longitudinal * (strain_t ** 2 + strain_n ** 2 + strain_z ** 2)
            + lame * (strain_t * strain_n + strain_t * strain_z + strain_n * strain_z))


def lateral_constrained_system(weights, lame, longitudinal, misfit_a, misfit_c, strain_t, fractions):
    """Lagrange system for (u_B, u_C, w, multiplier): normal strains of the two domains and shared c strain."""
    shape = weights.shape[:-1]
    matrix = np.zeros(shape + (4, 4))
    right = np.zeros(shape + (4,))
    for domain in (0, 1):
        weight, lam, lon = weights[..., domain], lame[..., domain], longitudinal[..., domain]
        elastic_t = strain_t[..., domain]
        matrix[..., domain, domain] = weight * lon
        matrix[..., domain, 2] = matrix[..., 2, domain] = weight * lam
        matrix[..., 2, 2] += weight * lon
        right[..., domain] = -weight * (-lon * misfit_a[..., domain] + lam * (elastic_t - misfit_c[..., domain]))
        right[..., 2] -= weight * (-lon * misfit_c[..., domain] + lam * (elastic_t - misfit_a[..., domain]))
        matrix[..., domain, 3] = matrix[..., 3, domain] = fractions[..., domain]
    return matrix, right


def clamped_lateral_energy(compositions, fractions, substrate_lattice_parameter):
    """Columnar domains on a clamping substrate (interfaces normal to the film); J/mol-f.u.

    compositions, fractions: shape (..., 2); a single domain falls back to the layered form.
    """
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    if compositions.shape[-1] == 1:
        return clamped_layered_energy(compositions, fractions, substrate_lattice_parameter)
    misfit_a, misfit_c, weights, reference_a = domain_properties(compositions, fractions)
    substrate_strain = np.asarray(substrate_lattice_parameter) / reference_a - 1.0
    strain_t = substrate_strain[..., None] - misfit_a
    lame, longitudinal = lame_constants(compositions)
    matrix, right = lateral_constrained_system(weights, lame, longitudinal, misfit_a, misfit_c,
                                               strain_t, fractions)
    right[..., 3] = substrate_strain
    solution = np.linalg.solve(matrix, right[..., None])[..., 0]
    strain_n = solution[..., :2] - misfit_a
    strain_z = solution[..., 2:3] - misfit_c
    density = isotropic_energy_density(strain_t, strain_n, strain_z, lame, longitudinal)
    return np.sum(weights * density, axis=-1)


ELASTIC_ENERGY_MODELS = {"incoherent": no_elastic_energy, "coherent": coherent_laminate_energy,
                         "clamped layered": clamped_layered_energy,
                         "clamped lateral": clamped_lateral_energy}

## 18. Free energy

Chemical part $G(x,T)$: Hirata's wurtzite Redlich–Kister (section 5) plus ideal cation mixing, J/mol-f.u.,
relative to the AlN–ScN wurtzite endmembers. One function spans both the buckled and flat branches. Two
**optional, flagged** corrections, off by default:

| correction | form | status |
|---|---|---|
| short-range order | $-\Omega^2x^2(1-x)^2/(zRT)$, $\Omega = L^{(0)}$, $z = 12$ | analytic high-temperature estimate, lowers $T_c$ ~9 % (exact Ising ~18 %); grows as $1/T$, **not valid far below $T_c$** |
| excess vibrational entropy | $\Delta S = \sigma x(1-x)$, 0.15 $k_B$/atom at $x = 0.5$ | **guessed** (typical 0.1–0.2 $k_B$/atom) |

Every function in sections 19–20 takes the free energy as an argument `gibbs_energy(x, T)`, vectorised in
`x`, so a corrected or entirely different $G$ can be passed in.

In [ ]:
CATION_COORDINATION_NUMBER = 12
EXCESS_VIBRATIONAL_ENTROPY_KB_PER_ATOM_AT_HALF = 0.15    # guess; typical 0.1-0.2


def configurational_entropy(composition):
    """Ideal cation-sublattice mixing entropy, J/(mol-f.u. K); accepts arrays."""
    x = np.asarray(composition, dtype=float)
    return -GAS_CONSTANT * (xlogy(x, x) + xlogy(1.0 - x, 1.0 - x))


def short_range_order_correction(composition, temperature):
    """Second-order (Onsager/TAP) SRO free energy, J/mol-f.u."""
    x = np.asarray(composition, dtype=float)
    interaction = REDLICH_KISTER_WURTZITE[0]
    return -interaction ** 2 * x ** 2 * (1.0 - x) ** 2 / (
        CATION_COORDINATION_NUMBER * GAS_CONSTANT * temperature)


def excess_vibrational_entropy(composition):
    """Guessed sigma*x(1-x) excess vibrational entropy of mixing, J/(mol-f.u. K)."""
    x = np.asarray(composition, dtype=float)
    per_formula_unit_at_half = (EXCESS_VIBRATIONAL_ENTROPY_KB_PER_ATOM_AT_HALF * GAS_CONSTANT
                                * ATOMS_PER_FORMULA_UNIT_WURTZITE)
    return 4.0 * per_formula_unit_at_half * x * (1.0 - x)


def chemical_gibbs_energy_of_mixing(composition, temperature, corrections=()):
    """Wurtzite G_mix(x, T), J/mol-f.u.; corrections: 'short_range_order', 'excess_vibrational_entropy'."""
    x = np.asarray(composition, dtype=float)
    energy = (excess_gibbs_energy(x, REDLICH_KISTER_WURTZITE)
              - temperature * configurational_entropy(x))
    if "short_range_order" in corrections:
        energy = energy + short_range_order_correction(x, temperature)
    if "excess_vibrational_entropy" in corrections:
        energy = energy - temperature * excess_vibrational_entropy(x)
    return energy


wurtzite_gibbs_energy = functools.partial(chemical_gibbs_energy_of_mixing, corrections=())

print("G_mix at x = 0.5, 673 K, meV/f.u.:")
for corrections in ((), ("short_range_order",), ("excess_vibrational_entropy",)):
    value = float(chemical_gibbs_energy_of_mixing(0.5, 673.15, corrections))
    print(f"  {' + '.join(corrections) or 'published (no correction)':30s} "
          f"{value * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f}")

## 19. Decomposition and stability at one composition

Each function below answers one question for **one** average composition $\bar x$ and temperature $T$, for a
given free energy `gibbs_energy(x, T)` and, for films on a substrate, a given substrate in-plane lattice
parameter `substrate_lattice_parameter` (Å). Phase diagrams (section 20) are built by calling them on a grid
of $(\bar x, T)$; a composition-graded film on a graded substrate is handled by passing a different substrate
value at each $\bar x$.

**Decomposition (binodal).** The film either stays homogeneous or splits into two domains $x_B < \bar x < x_C$
with fraction $f = (x_C-\bar x)/(x_C-x_B)$ of $x_B$ (lever rule). For each elastic model of section 17,

$$\Delta G_\mathrm{decomp} = \big[G(\bar x) + E_\mathrm{el}(\bar x)\big] - \min_{x_B, x_C}\big[f\,G(x_B) + (1-f)\,G(x_C) + E_\mathrm{el}(x_B, x_C, f)\big].$$

$\Delta G_\mathrm{decomp} > 0$ means the film lowers its free energy by decomposing; the binodal is where it
becomes zero. The minimum is found by evaluating **every pair** on a 48 × 48 grid spanning
$0 \le x_B < \bar x$ and $\bar x < x_C \le 1$ — so a partner in a second gap is found as readily as one in
the first — and then refining the best pair with bounded Nelder–Mead (derivative-free, because the
minimum over interface angles makes the objective kinked). For incoherent domains this is exactly the
common tangent; for coherent ones it is not, because $E_\mathrm{el}$ is not linear in $f$.

**Stability (spinodal).** The same comparison for an infinitesimal split $x_B, x_C = \bar x \mp \delta$,
$f = 1/2$, gives an effective curvature
$G''_\mathrm{eff} = 2\big[\tfrac12 G(\bar x-\delta)+\tfrac12 G(\bar x+\delta) + E_\mathrm{el}(\text{split}) - G(\bar x) - E_\mathrm{el}(\bar x)\big]/\delta^2$,
$\delta = 10^{-3}$. The film is unstable where $G''_\mathrm{eff} < 0$. Without elastic energy this is
$\partial^2G/\partial x^2$; for the coherent laminate it is Cahn's $\partial^2G/\partial x^2 + 2\eta^2YV_m$
minimised over orientation.

Two-domain states only: a three-domain state is not searched.

In [ ]:
PARTNER_GRID_POINTS = 48
FLUCTUATION_AMPLITUDE = 1e-3


@dataclasses.dataclass
class DecompositionResult:
    """Best two-domain state at one average composition; energies in J/mol-f.u."""
    average_composition: float
    driving_force: float                  # > 0: decomposition lowers G; 0: stays homogeneous
    domain_compositions: tuple            # (x_B, x_C); both equal to the average if homogeneous
    fraction_of_lower_domain: float       # f of x_B


def two_domain_free_energy(gibbs_energy, temperature, lower, upper, average, elastic_energy):
    """f G(x_B) + (1-f) G(x_C) + E_el for arrays of pairs (x_B, x_C) at one average composition."""
    lower, upper = np.asarray(lower, dtype=float), np.asarray(upper, dtype=float)
    fraction = (upper - average) / (upper - lower)
    compositions = np.stack([lower, upper], axis=-1)
    fractions = np.stack([fraction, 1.0 - fraction], axis=-1)
    return (fraction * gibbs_energy(lower, temperature) + (1.0 - fraction) * gibbs_energy(upper, temperature)
            + elastic_energy(compositions, fractions))


def homogeneous_free_energy(gibbs_energy, temperature, average, elastic_energy):
    """G + E_el of the undecomposed film at the average composition, J/mol-f.u."""
    elastic = elastic_energy(np.array([[average]]), np.array([[1.0]]))[0]
    return float(gibbs_energy(np.array(average), temperature)) + float(elastic)


def pair_objective(pair, gibbs_energy, temperature, average, elastic_energy):
    """Scalar version of two_domain_free_energy for the refinement step."""
    if pair[0] > average - 1e-9 or pair[1] < average + 1e-9:     # a domain with zero fraction
        return homogeneous_free_energy(gibbs_energy, temperature, average, elastic_energy)
    return float(two_domain_free_energy(gibbs_energy, temperature, np.array([pair[0]]),
                                        np.array([pair[1]]), average, elastic_energy)[0])


def decomposition(gibbs_energy, temperature, average_composition, elastic_energy):
    """Best two-domain state of a film of average composition x (0 < x < 1); returns DecompositionResult.

    gibbs_energy(x, T): J/mol-f.u., vectorised in x. elastic_energy(compositions, fractions): a section-17
    model (substrate already bound), arrays of shape (..., domains) -> (...).
    """
    average = float(average_composition)
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_energy)
    lower, upper = np.meshgrid(np.linspace(0.0, average, PARTNER_GRID_POINTS, endpoint=False),
                               np.linspace(1.0, average, PARTNER_GRID_POINTS, endpoint=False), indexing="ij")
    energies = two_domain_free_energy(gibbs_energy, temperature, lower, upper, average, elastic_energy)
    best = np.unravel_index(np.argmin(energies), energies.shape)
    refined = minimize(pair_objective, [lower[best], upper[best]], method="Nelder-Mead",
                       args=(gibbs_energy, temperature, average, elastic_energy),
                       bounds=[(0.0, average), (average, 1.0)], options={"xatol": 1e-5, "fatol": 1e-3})
    if refined.fun >= homogeneous:
        return DecompositionResult(average, 0.0, (average, average), 1.0)
    lower_x, upper_x = refined.x
    return DecompositionResult(average, homogeneous - refined.fun, (lower_x, upper_x),
                               (upper_x - average) / (upper_x - lower_x))


def stability(gibbs_energy, temperature, average_composition, elastic_energy,
              amplitude=FLUCTUATION_AMPLITUDE):
    """Effective curvature (J/mol-f.u.) against a small +-amplitude two-domain split; < 0 is unstable."""
    average = float(average_composition)
    amplitude = min(amplitude, 0.5 * average, 0.5 * (1.0 - average))
    split = two_domain_free_energy(gibbs_energy, temperature, np.array([average - amplitude]),
                                   np.array([average + amplitude]), average, elastic_energy)[0]
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_energy)
    return 2.0 * (split - homogeneous) / amplitude ** 2

The functions to call, one per case. Clamped films take `geometry = "layered"`, `"lateral"`, or `"lowest"`
(whichever decomposes more readily).

In [ ]:
def bind_substrate(geometry, substrate_lattice_parameter):
    """The section-17 clamped model for a geometry, with the substrate lattice parameter fixed."""
    return functools.partial(ELASTIC_ENERGY_MODELS[f"clamped {geometry}"],
                             substrate_lattice_parameter=substrate_lattice_parameter)


def incoherent_decomposition(gibbs_energy, temperature, average_composition):
    """Dislocated domains, each at its own lattice; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, no_elastic_energy)


def coherent_decomposition(gibbs_energy, temperature, average_composition):
    """Free-standing coherent domains, softest interface orientation; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, coherent_laminate_energy)


def clamped_decomposition(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                          geometry="lowest"):
    """Film clamped to a substrate (in-plane lattice parameter, Angstrom); returns DecompositionResult."""
    if geometry != "lowest":
        return decomposition(gibbs_energy, temperature, average_composition,
                             bind_substrate(geometry, substrate_lattice_parameter))
    return max((clamped_decomposition(gibbs_energy, temperature, average_composition,
                                      substrate_lattice_parameter, option) for option in ("layered", "lateral")),
               key=operator.attrgetter("driving_force"))


def incoherent_stability(gibbs_energy, temperature, average_composition):
    """d2G/dx2, J/mol-f.u.; < 0 inside the chemical spinodal."""
    return stability(gibbs_energy, temperature, average_composition, no_elastic_energy)


def coherent_stability(gibbs_energy, temperature, average_composition):
    """Cahn coherent-spinodal curvature, softest orientation, J/mol-f.u."""
    return stability(gibbs_energy, temperature, average_composition, coherent_laminate_energy)


def clamped_stability(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                      geometry="lowest"):
    """Spinodal curvature of a clamped film, J/mol-f.u.; 'lowest' takes the less stable geometry."""
    if geometry != "lowest":
        return stability(gibbs_energy, temperature, average_composition,
                         bind_substrate(geometry, substrate_lattice_parameter))
    return min(clamped_stability(gibbs_energy, temperature, average_composition,
                                 substrate_lattice_parameter, option) for option in ("layered", "lateral"))

In [ ]:
_average = 0.3
_lattice = float(lattice_parameters(_average)[0])
_split = (np.array([[_average - 1e-3, _average + 1e-3]]), np.array([[0.5, 0.5]]))
_single = (np.array([[_average]]), np.array([[1.0]]))
check_value(float(clamped_layered_energy(*_split, _lattice)[0] - clamped_layered_energy(*_single, _lattice)[0]),
            float(coherent_laminate_energy(*_split, angles=np.array([0.0]))[0]), 1e-3,
            "small split: clamped layered (a_sub = a(x)) equals coherent basal laminate")
check_value(float(clamped_lateral_energy(*_split, _lattice)[0] - clamped_layered_energy(*_single, _lattice)[0]),
            float(coherent_laminate_energy(*_split, angles=np.array([0.5 * math.pi]))[0]), 1e-3,
            "small split: clamped lateral (a_sub = a(x)) equals coherent prism laminate")
_mesh = np.linspace(0.0, 1.0, 4001)
_energies = wurtzite_gibbs_energy(_mesh, 673.15)
_hull = lower_hull_indices(_mesh, _energies)
check_value(incoherent_decomposition(wurtzite_gibbs_energy, 673.15, 0.3).driving_force,
            float(wurtzite_gibbs_energy(np.array(0.3), 673.15) - np.interp(0.3, _mesh[_hull], _energies[_hull])),
            1e-4, "incoherent decomposition equals the convex-hull common tangent")

## 20. Results

Phase boundaries are contours of the section-19 functions evaluated on a grid of $(\bar x, T)$: the binodal is
where $\Delta G_\mathrm{decomp}$ first exceeds 1 J/mol-f.u. (0.01 meV/f.u.), the spinodal where
$G''_\mathrm{eff}$ changes sign. Both capture any number of separate regions.

The substrate argument of the map functions may be a number or a function of $\bar x$ (graded substrate).
Partial relaxation of a thick film is expressed through such a function: the film keeps the fraction
$\sqrt{f(h)}$ of its misfit to the substrate, $f(h)$ being Hirata's relaxation factor of the energy (section 7).

In [ ]:
GROWTH_TEMPERATURE = 673.15
GROWTH_TEMPERATURES = (573.15, 673.15, 723.15)
DECOMPOSITION_THRESHOLD = 1.0      # J/mol-f.u.
REAL_FILM_THICKNESS_NM = 1000.0


def effective_substrate_lattice_parameter(substrate_lattice_parameter, film_thickness_nm):
    """Function x -> in-plane lattice imposed on a partly relaxed film of average composition x."""
    retained = math.sqrt(strain_relaxation_factor(film_thickness_nm, CRITICAL_THICKNESS_NM))
    return functools.partial(partially_relaxed_lattice, substrate_lattice_parameter=substrate_lattice_parameter,
                             retained_misfit=retained)


def partially_relaxed_lattice(composition, substrate_lattice_parameter, retained_misfit):
    own = float(lattice_parameters(composition)[0])
    return own * (1.0 + retained_misfit * (substrate_lattice_parameter / own - 1.0))


def substrate_at(substrate_lattice_parameter, composition):
    """Substrate lattice parameter for one average composition: a number or a function of x."""
    if callable(substrate_lattice_parameter):
        return substrate_lattice_parameter(composition)
    return substrate_lattice_parameter


def phase_map(function, gibbs_energy, temperatures, compositions, substrate_lattice_parameter=None, **options):
    """function(gibbs_energy, T, x[, substrate], **options) on a (T, x) grid -> array (len(T), len(x)).

    Decomposition functions contribute their driving force; stability functions their curvature.
    """
    values = np.zeros((len(temperatures), len(compositions)))
    for row, temperature in enumerate(temperatures):
        for column, composition in enumerate(compositions):
            arguments = (gibbs_energy, temperature, composition)
            if substrate_lattice_parameter is not None:
                arguments += (substrate_at(substrate_lattice_parameter, composition),)
            result = function(*arguments, **options)
            values[row, column] = getattr(result, "driving_force", result)
    return values


def critical_temperature(stability_function, gibbs_energy, compositions, substrate_lattice_parameter=None,
                         low=100.0, high=9000.0, iterations=16):
    """Highest T at which the stability function is negative somewhere on the composition grid; (T, x).

    (nan, nan): stable at `low`; (inf, nan): still unstable at `high`.
    """
    if not np.any(phase_map(stability_function, gibbs_energy, [low], compositions,
                            substrate_lattice_parameter) < 0.0):
        return float("nan"), float("nan")
    if np.any(phase_map(stability_function, gibbs_energy, [high], compositions,
                        substrate_lattice_parameter) < 0.0):
        return float("inf"), float("nan")
    for _ in range(iterations):
        middle = 0.5 * (low + high)
        unstable = np.any(phase_map(stability_function, gibbs_energy, [middle], compositions,
                                    substrate_lattice_parameter) < 0.0)
        low, high = (middle, high) if unstable else (low, middle)
    curvature = phase_map(stability_function, gibbs_energy, [low], compositions, substrate_lattice_parameter)[0]
    return low, float(compositions[int(np.argmin(curvature))])


SUBSTRATE_CASES = {
    "on AlN, fully clamped": SUBSTRATE_IN_PLANE_LATTICE["AlN"],
    f"on AlN, {REAL_FILM_THICKNESS_NM:.0f} nm": effective_substrate_lattice_parameter(
        SUBSTRATE_IN_PLANE_LATTICE["AlN"], REAL_FILM_THICKNESS_NM),
}

### Critical temperatures

In [ ]:
CRITICAL_COMPOSITIONS = np.linspace(0.01, 0.99, 197)
BUCKLED_COMPOSITIONS = CRITICAL_COMPOSITIONS[CRITICAL_COMPOSITIONS < TRANSITION_COMPOSITION_COFIT - 0.05]
CRITICAL_CASES = {"incoherent": (incoherent_stability, None),
                  "coherent, free-standing": (coherent_stability, None)}
CRITICAL_CASES.update({f"clamped {name}": (clamped_stability, substrate)
                       for name, substrate in SUBSTRATE_CASES.items()})
CRITICAL_TEMPERATURES = {name: (critical_temperature(function, wurtzite_gibbs_energy, CRITICAL_COMPOSITIONS,
                                                     substrate),
                                critical_temperature(function, wurtzite_gibbs_energy, BUCKLED_COMPOSITIONS,
                                                     substrate))
                         for name, (function, substrate) in CRITICAL_CASES.items()}
print(f"{'':32s} {'all x':>22} {'x < x0 - 0.05':>22}")
for name, results in CRITICAL_TEMPERATURES.items():
    print(f"  {name:30s} " + " ".join(
        f"{'no gap':>22}" if math.isnan(temperature) else f"{'unstable at all T':>22}" if math.isinf(temperature)
        else f"{temperature:8.0f} K at x = {composition:.3f}" for temperature, composition in results))
check_value(CRITICAL_TEMPERATURES["incoherent"][0][0], 6971.8, 0.005, "incoherent T_c vs section 6")

### Phase diagram

Solid: binodal; dotted: spinodal. Left: free-standing (bulk, or film fully relaxed from its substrate). Middle
and right: films on AlN, fully clamped and as a 1 µm film; `geometry = "lowest"`. Grid: 250 K × 0.025 in $x$
for the binodal, 100 K × 0.005 for the spinodal.

The clamped panels show narrow composition bands that stay unstable up to the top of the plot. There the
clamping energy — up to ~200 meV/f.u. for a fully clamped film — changes faster with $x$ than the entropy can
offset, through the misfit, the softening $E(x)$, and the steep $a(x)$ near $x_0$; the last depends on the
assumed transition width. A fully clamped film is only meaningful below $h_c \approx 2$ nm.

In [ ]:
BINODAL_TEMPERATURES = np.arange(300.0, 7400.0, 250.0)
BINODAL_COMPOSITIONS = np.arange(0.0125, 0.99, 0.025)
SPINODAL_TEMPERATURES = np.arange(300.0, 7400.0, 100.0)
SPINODAL_COMPOSITIONS = np.linspace(0.005, 0.995, 199)

PHASE_MAPS = {"incoherent": (phase_map(incoherent_decomposition, wurtzite_gibbs_energy, BINODAL_TEMPERATURES,
                                       BINODAL_COMPOSITIONS),
                             phase_map(incoherent_stability, wurtzite_gibbs_energy, SPINODAL_TEMPERATURES,
                                       SPINODAL_COMPOSITIONS)),
              "coherent": (phase_map(coherent_decomposition, wurtzite_gibbs_energy, BINODAL_TEMPERATURES,
                                     BINODAL_COMPOSITIONS),
                           phase_map(coherent_stability, wurtzite_gibbs_energy, SPINODAL_TEMPERATURES,
                                     SPINODAL_COMPOSITIONS))}
for name, substrate in SUBSTRATE_CASES.items():
    PHASE_MAPS[name] = (phase_map(clamped_decomposition, wurtzite_gibbs_energy, BINODAL_TEMPERATURES,
                                  BINODAL_COMPOSITIONS, substrate),
                        phase_map(clamped_stability, wurtzite_gibbs_energy, SPINODAL_TEMPERATURES,
                                  SPINODAL_COMPOSITIONS, substrate))

In [ ]:
def draw_boundaries(axis, name, colour, label):
    """Binodal (solid) and spinodal (dotted) contours of one PHASE_MAPS entry."""
    driving_force, curvature = PHASE_MAPS[name]
    # sqrt: the driving force grows quadratically from the binodal, so its root interpolates linearly
    axis.contour(BINODAL_COMPOSITIONS, BINODAL_TEMPERATURES, np.sqrt(driving_force),
                 levels=[math.sqrt(DECOMPOSITION_THRESHOLD)], colors=colour, linestyles="-")
    axis.contour(SPINODAL_COMPOSITIONS, SPINODAL_TEMPERATURES, curvature, levels=[0.0], colors=colour,
                 linestyles=":")
    axis.plot([], [], "-", color=colour, label=f"{label}, binodal")
    axis.plot([], [], ":", color=colour, label=f"{label}, spinodal")


if HAVE_MATPLOTLIB:
    figure, axes = plt.subplots(1, 3, figsize=(16, 5.2), sharey=True)
    draw_boundaries(axes[0], "incoherent", "k", "incoherent")
    draw_boundaries(axes[0], "coherent", "tab:red", "coherent")
    draw_boundaries(axes[1], "incoherent", "grey", "incoherent")
    draw_boundaries(axes[1], "on AlN, fully clamped", "tab:blue", "clamped on AlN")
    draw_boundaries(axes[2], "incoherent", "grey", "incoherent")
    draw_boundaries(axes[2], f"on AlN, {REAL_FILM_THICKNESS_NM:.0f} nm", "tab:green", "AlN, 1 um")
    titles = ("free-standing", "film fully clamped to AlN", f"{REAL_FILM_THICKNESS_NM:.0f} nm film on AlN")
    for axis, title in zip(axes, titles):
        axis.axhspan(GROWTH_TEMPERATURES[0], GROWTH_TEMPERATURES[-1], color="tab:olive", alpha=0.2)
        axis.axvline(TRANSITION_COMPOSITION_COFIT, color="grey", ls="--", lw=0.8)
        axis.set_title(title); axis.set_xlabel("Sc cation fraction $x$"); axis.set_xlim(0, 1)
        axis.legend(fontsize=8, loc="upper right"); axis.grid(alpha=0.25)
    axes[0].set_ylabel("Temperature (K)")
    figure.suptitle("Wurtzite Al$_{1-x}$Sc$_x$N, Hirata free energy, no corrections "
                    "(band: growth window 300-450 C; dashed: x$_0$)")
    plt.tight_layout(); plt.show()

### Decomposition driving force

$\Delta G_\mathrm{decomp}$ at growth temperatures, meV/f.u., with the domain compositions of the best state
at 400 °C.

In [ ]:
DRIVING_FORCE_COMPOSITIONS = np.linspace(0.02, 0.98, 49)
DRIVING_FORCE_CASES = {
    "incoherent": (incoherent_decomposition, None),
    "coherent, free-standing": (coherent_decomposition, None),
    "clamped on AlN": (clamped_decomposition, SUBSTRATE_CASES["on AlN, fully clamped"]),
    "1 um on AlN": (clamped_decomposition, SUBSTRATE_CASES[f"on AlN, {REAL_FILM_THICKNESS_NM:.0f} nm"]),
}
DRIVING_FORCE = {(name, temperature): phase_map(function, wurtzite_gibbs_energy, [temperature],
                                               DRIVING_FORCE_COMPOSITIONS, substrate)[0]
                 for name, (function, substrate) in DRIVING_FORCE_CASES.items()
                 for temperature in GROWTH_TEMPERATURES}

print(f"Best state at {GROWTH_TEMPERATURE - 273.15:.0f} C: driving force (meV/f.u.) and domains x_B + x_C\n")
print(f"{'x':>6} " + " ".join(f"{name:>30}" for name in DRIVING_FORCE_CASES))
for composition in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8):
    cells = []
    for function, substrate in DRIVING_FORCE_CASES.values():
        arguments = (wurtzite_gibbs_energy, GROWTH_TEMPERATURE, composition)
        if substrate is not None:
            arguments += (substrate_at(substrate, composition),)
        result = function(*arguments)
        lower, upper = result.domain_compositions
        cells.append(f"{result.driving_force * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:7.1f} "
                     f"({lower:.3f} + {upper:.3f})")
    print(f"{composition:6.2f} " + " ".join(f"{cell:>30}" for cell in cells))

In [ ]:
if HAVE_MATPLOTLIB:
    figure, axis = plt.subplots(figsize=(10.0, 5.0))
    colours = {"incoherent": "k", "coherent, free-standing": "tab:red", "clamped on AlN": "tab:blue",
               "1 um on AlN": "tab:green"}
    styles = {573.15: ":", 673.15: "-", 723.15: "--"}
    for (name, temperature), values in DRIVING_FORCE.items():
        axis.plot(DRIVING_FORCE_COMPOSITIONS, values * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE, styles[temperature],
                  color=colours[name], label=f"{name}, {temperature - 273.15:.0f} C")
    axis.axvline(TRANSITION_COMPOSITION_COFIT, color="grey", ls="--", lw=0.8)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel(r"$\Delta G_\mathrm{decomp}$ (meV/f.u.)")
    axis.set_title("Decomposition driving force at growth temperatures (Hirata free energy, no corrections)")
    axis.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1.0)); axis.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

## 21. Identifiability

With a linear misfit, Hirata's substrate strain energy is a shift of $L^{(0)}$ plus an endmember tilt (section 7).
With the real $a(x)$ the clamped energy on the buckled branch is fitted to a line plus
$x(1-x)[\Delta L^{(0)} + \Delta L^{(1)}(1-2x)]$. If the fit is near-exact, a strain term and a change of $L^{(0)}$ cannot
be told apart from one measurement, and coherency adds a third contribution of the same kind. Separating
them needs phase boundaries at **two or more thicknesses or substrates** — a design requirement for Library 1.

In [ ]:
STRAIN_FIT_COMPOSITIONS = np.linspace(0.0, 0.45, 46)    # buckled branch, film compositions


def substrate_strain_energy(composition, substrate_lattice_parameter, film_thickness_nm=None):
    """Hirata's clamped-film energy with relaxation factor f(h) (None: fully clamped), J/mol-f.u."""
    energy = clamped_layered_energy(np.atleast_1d(composition)[..., None], np.ones((np.size(composition), 1)),
                                    substrate_lattice_parameter)
    if film_thickness_nm is not None:
        energy = energy * strain_relaxation_factor(film_thickness_nm, CRITICAL_THICKNESS_NM)
    return energy


def strain_as_interaction_parameters(substrate_lattice_parameter, film_thickness_nm):
    """(dL0, dL1) in J/mol-f.u. and the relative rms residual of the fit, 0 <= x <= 0.45."""
    x = STRAIN_FIT_COMPOSITIONS
    energies = substrate_strain_energy(x, substrate_lattice_parameter, film_thickness_nm)
    basis = np.stack([np.ones_like(x), x, x * (1.0 - x), x * (1.0 - x) * (1.0 - 2.0 * x)], axis=1)
    coefficients, *_ = np.linalg.lstsq(basis, energies, rcond=None)
    excess = energies - basis[:, :2] @ coefficients[:2]
    residual = energies - basis @ coefficients
    return coefficients[2:], math.sqrt(np.mean(residual ** 2)) / np.max(np.abs(excess))


print(f"{'substrate':>9} {'h (nm)':>7} {'dL0 (meV/f.u.)':>15} {'dL1':>8} {'rel. rms':>9}   (L0 = "
      f"{REDLICH_KISTER_WURTZITE[0] * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:.0f} meV/f.u.)")
for substrate in ("AlN", "GaN"):
    for thickness_nm in (10.0, 100.0, 1000.0):
        (delta_l0, delta_l1), relative = strain_as_interaction_parameters(
            SUBSTRATE_IN_PLANE_LATTICE[substrate], thickness_nm)
        print(f"{substrate:>9} {thickness_nm:7.0f} {delta_l0 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:15.1f} "
              f"{delta_l1 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f} {relative:9.4f}")

## 22. Interface for the defect model

The defect model's chemical potentials need the free energy of the **homogeneous** film. Relative to the
unstrained AlN–ScN wurtzite endmembers, J/mol-f.u.; with a substrate it includes the biaxial clamping energy.

In [ ]:
def homogeneous_film_gibbs_energy(gibbs_energy, temperature, composition, substrate_lattice_parameter=None):
    """G of an undecomposed film; substrate_lattice_parameter: None (free) or Angstrom."""
    composition = np.asarray(composition, dtype=float)
    energy = gibbs_energy(composition, temperature)
    if substrate_lattice_parameter is None:
        return energy
    return energy + substrate_strain_energy(composition, substrate_lattice_parameter).reshape(np.shape(energy))


for substrate in (None, SUBSTRATE_IN_PLANE_LATTICE["AlN"]):
    value = float(homogeneous_film_gibbs_energy(wurtzite_gibbs_energy, GROWTH_TEMPERATURE, 0.25, substrate))
    print(f"  G(x = 0.25, 673 K), substrate {substrate}:  "
          f"{value * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:7.1f} meV/f.u.")

## 23. Check summary

Every check recorded since section 9, including those of the strain model.

In [ ]:
summarise_checks()